In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports and Plot Style ===

# --- Dependency check: install missing packages into the active kernel ---
import importlib, subprocess, sys

_REQUIRED_PKGS = {
    "sklearn":    "scikit-learn",
    "plotly":     "plotly",
    "ipywidgets": "ipywidgets",
}
for _mod, _pkg in _REQUIRED_PKGS.items():
    if importlib.util.find_spec(_mod) is None:
        print(f"Installing {_pkg} …", flush=True)
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", _pkg, "--quiet"],
            stdout=subprocess.DEVNULL,
        )
        print(f"  {_pkg} installed ✓")

del _mod, _pkg, _REQUIRED_PKGS

from pathlib import Path
import re
import json
import uuid
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go

# --- Plot style (KI-Sort conventions) ---
_FW        = 16 / 2.54   # full LaTeX \textwidth in inches
_FONT_SIZE = 10          # base font size at \textwidth

plt.style.use("default")
plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "font.size":       _FONT_SIZE,
    "axes.titlesize":  _FONT_SIZE + 1,
    "axes.labelsize":  _FONT_SIZE,
    "legend.fontsize": _FONT_SIZE - 1,
    "savefig.dpi":     150,
    "figure.dpi":      110,
    "axes.grid":       True,
    "grid.alpha":      0.25,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "legend.frameon":  False,
    "mathtext.fontset": "dejavusans",
})

NOTEBOOK_THEME_CSS = """
<style>
.dv-title { font-weight: 700; color: #1f2937; margin: 0 0 6px 0; font-size: 16px; }
.dv-subtitle { color: #4b5563; margin: 0 0 10px 0; font-size: 12px; }
.dv-status { padding: 6px 10px; border-radius: 6px; border: 1px solid #d1d5db;
             background: #f8fafc; color: #111827; font-size: 12px; }
</style>
"""
display(HTML(NOTEBOOK_THEME_CSS))
print(f"Imports complete ✓   (_FW={_FW:.3f} in, _FONT_SIZE={_FONT_SIZE} pt, font=DejaVu Sans)")

In [ ]:
# === Cell 3: Config and Helpers ===

# --- Paths (all paths are set here) ---
PROJECT_NAME = "Peak_Finder"

BASE_DIR  = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_NIR_Dataset/")
SAVE_DIR  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/04_IR_Preprocessing/peak_analysis_results")

# GT_PATH is either a CSV file (region-aware GT: body/label/cap) or a directory
# (GT is then derived from filenames by the Filename-Derived Ground Truth cell).
GT_PATH   = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_NIR_Dataset/spectra_ground_truth_v8.csv")

RANDOM_STATE   = 42
DEFAULT_FOLDER = "raw_spectra"

# --- Metadata store: controls how spectra are labelled and coloured ---
METADATA = {
    "gt_lookup":  {},     # dict[stem, class_label]
    "gt_source":  None,   # "csv" | "filename" | None
    "has_regions": False, # True when filenames carry _body/_label/_cap suffixes
    "gt_csv":     None,   # raw GT DataFrame (or None)
}

# --- Ground truth: CSV file or directory ---
if GT_PATH.is_file():
    _gt = pd.read_csv(GT_PATH, sep=";")
    _gt.columns = [c.strip() for c in _gt.columns]
    _gt = _gt.drop(
        columns=[c for c in _gt.columns if c in ("label_description", "") or c.startswith("Unnamed")],
        errors="ignore",
    )
    _gt = _gt.dropna(subset=["filename"], how="all")
    _gt = _gt[_gt["filename"].astype(str).str.strip().ne("")]
    _gt["filename"]    = _gt["filename"].astype(str).str.strip()
    _gt["body_class"]  = _gt["body_class"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})
    _gt["label_class"] = _gt["label_class"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})
    _gt["cap_class"]   = _gt["cap_class"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})

    _lk = {}
    for _, _row in _gt.iterrows():
        _base = _row["filename"]
        for _part, _col in [("body", "body_class"), ("label", "label_class"), ("cap", "cap_class")]:
            if pd.notna(_row.get(_col)):
                _lk[f"{_base}_{_part}"] = str(_row[_col])
    METADATA["gt_lookup"]   = _lk
    METADATA["gt_source"]   = "csv"
    METADATA["has_regions"] = True
    METADATA["gt_csv"]      = _gt
    print(f"Ground truth loaded from CSV: {len(_gt)} objects ({GT_PATH.name})")
    print(f"  Per-stem lookup entries: {len(_lk)}   |   has_regions=True")
elif GT_PATH.is_dir():
    print(f"GT_PATH points to a DIRECTORY:\n  {GT_PATH}")
    print("  -> Run Cell 4 to derive ground truth from filenames in this directory.")
    print("     Until then, all spectra will be labelled 'unknown' (single colour).")
else:
    print(f"GT_PATH not found: {GT_PATH}")
    print("  -> Run Cell 4 for filename-derived GT, or fix the path above.")

# --- Class colour palette ---
_FIXED_CLASS_COLORS = {
    "HDPE":  "#1f4e8c",   # dark blue
    "LDPE":  "#7bafd4",   # light blue
    "PE":    "#1f77b4",   # standard blue (legacy CSV class)
    "PET":   "#ff7f0e",   # orange
    "PP":    "#2ca02c",   # green
    "PS":    "#d62728",   # red
    "Paper": "#8c564b",   # brown
}
_SEED_COLORS = [
    "#9467bd", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5",
]

def build_palette(classes):
    """Assign one colour per class, using fixed assignments where available."""
    palette = {}
    seed_idx = 0
    for cls in sorted(classes):
        if cls in _FIXED_CLASS_COLORS:
            palette[cls] = _FIXED_CLASS_COLORS[cls]
        else:
            palette[cls] = _SEED_COLORS[seed_idx % len(_SEED_COLORS)]
            seed_idx += 1
    return palette

_all_classes = sorted(set(METADATA["gt_lookup"].values())) if METADATA["gt_lookup"] else []
CLASS_PALETTE = build_palette(_all_classes) if _all_classes else {}

if CLASS_PALETTE:
    print(f"Class palette ({len(CLASS_PALETTE)} classes): {list(CLASS_PALETTE.keys())}")
else:
    print("No class palette yet (run Cell 4 or select a folder with GT-matched data).")

# --- Marker and symbol maps for region-differentiated plots ---
PART_MARKERS_MPL    = {"body": "o", "label": "s", "cap": "^"}
PART_SYMBOLS_PLOTLY = {"body": "circle", "label": "square", "cap": "diamond"}
PART_LABELS         = {"body": "Body", "label": "Label", "cap": "Cap"}

# --- Shared helpers ---
def get_subfolders(base_dir):
    if not base_dir.is_dir():
        return []
    return sorted([d.name for d in base_dir.iterdir() if d.is_dir()], key=str.lower)


def parse_filename(fname):
    """Parse 'HDPE_obj_01_body' -> ('HDPE_obj_01', 'body'). part=None if no suffix."""
    stem = Path(fname).stem
    m = re.search(r"_(body|label|cap)$", stem, flags=re.IGNORECASE)
    if m:
        return stem[:m.start()], m.group(1).lower()
    return stem, None


def get_gt_class(stem):
    return METADATA["gt_lookup"].get(stem)


def lookup_gt_class(base_name, part, gt_df=None):
    """Look up the GT class of base_name + part in the GT CSV; None if absent."""
    if gt_df is None:
        gt_df = METADATA.get("gt_csv")
    if gt_df is None:
        return None
    rows = gt_df[gt_df["filename"] == base_name]
    if rows.empty:
        return None
    row = rows.iloc[0]
    col = {"body": "body_class", "label": "label_class", "cap": "cap_class"}.get(part)
    if col and pd.notna(row[col]):
        return str(row[col])
    return None


def load_spectra_matrix(data_dir, include_parts=None):
    """
    Read spectra files from *data_dir*, interpolated onto the first file's axis.
    With has_regions and include_parts, only files with those suffixes are loaded.
    Returns (X, meta_list, wn_ref).
    """
    has_regions = METADATA["has_regions"]

    if has_regions and include_parts:
        allowed_suffixes = tuple(include_parts)
        files = sorted(
            f for f in data_dir.iterdir()
            if f.is_file() and any(f.stem.lower().endswith(s) for s in allowed_suffixes)
        )
    else:
        files = sorted(
            f for f in data_dir.iterdir()
            if f.is_file() and not f.name.startswith(".")
        )

    if not files:
        if has_regions and include_parts:
            raise FileNotFoundError(
                f"No spectra files found in {data_dir} for parts: {list(include_parts)}"
            )
        raise FileNotFoundError(f"No spectra files found in {data_dir}")

    spectra, meta, wn_ref = [], [], None
    parse_errors = []
    for f in files:
        base_name, part = parse_filename(f.name)
        try:
            df = pd.read_csv(f, sep=";", header=None, names=["Wavenumber", "Reflectance"])
            df["Wavenumber"]  = pd.to_numeric(df["Wavenumber"], errors="coerce")
            df["Reflectance"] = pd.to_numeric(df["Reflectance"], errors="coerce")
            df = df.dropna().sort_values("Wavenumber")
            if df.empty:
                parse_errors.append(f.name)
                continue
        except Exception:
            parse_errors.append(f.name)
            continue

        wn = df["Wavenumber"].to_numpy()
        y  = df["Reflectance"].to_numpy()

        if wn_ref is None:
            wn_ref = wn
        elif len(wn) != len(wn_ref) or not np.allclose(wn, wn_ref):
            y = np.interp(wn_ref, wn, y)

        spectra.append(y)
        meta.append({"filename": f.name, "base_name": base_name, "part": part})

    if parse_errors:
        print(f"  Skipped {len(parse_errors)} unparseable file(s): {parse_errors[:5]}"
              + (" ..." if len(parse_errors) > 5 else ""))

    if not spectra:
        raise FileNotFoundError(f"No parseable spectra found in {data_dir.name}")

    return np.vstack(spectra), meta, wn_ref


def build_plot_df(X, meta):
    """Attach a 'plot_class' column from METADATA['gt_lookup']; 'unknown' where absent."""
    meta_df = pd.DataFrame(meta)
    has_regions = METADATA["has_regions"]
    gt_lookup = METADATA["gt_lookup"]

    classes = []
    for _, row in meta_df.iterrows():
        stem = Path(row["filename"]).stem
        cls = gt_lookup.get(stem)
        if cls is None and has_regions:
            cls = lookup_gt_class(row["base_name"], row["part"])
        classes.append(cls if cls is not None else "unknown")

    meta_df["plot_class"] = classes

    n_unknown = (meta_df["plot_class"] == "unknown").sum()
    if 0 < n_unknown < len(meta_df):
        print(f"  Note: {n_unknown} spectrum/spectra have no GT class (labelled 'unknown').")

    return X, meta_df


def _ensure_palette(classes):
    """Add colours to CLASS_PALETTE for any missing class (iterable or DataFrame column)."""
    global CLASS_PALETTE
    cls_set = set(str(c) for c in classes)
    missing = cls_set - set(CLASS_PALETTE.keys())
    if missing:
        CLASS_PALETTE = build_palette(sorted(set(CLASS_PALETTE.keys()) | cls_set))


def build_mpl_legends(ax, plot_df, palette, show_parts=True, fontsize=None):
    """
    Draw one combined legend (class colours, plus region markers if available)
    inside the axes at loc="best". `fontsize` overrides the default legend size.
    """
    has_regions = METADATA["has_regions"]
    if fontsize is None:
        fontsize = _FONT_SIZE - 1

    present_classes = sorted(plot_df["plot_class"].unique())
    handles = [
        Line2D([0], [0], marker="o", color="w", label=cls,
               markerfacecolor=palette.get(cls, "#7f7f7f"),
               markeredgecolor="black", markersize=8)
        for cls in present_classes
    ]

    if show_parts and has_regions:
        active_parts = sorted(plot_df["part"].dropna().unique())
        handles += [
            Line2D([0], [0], marker=PART_MARKERS_MPL[p], color="w",
                   label=PART_LABELS.get(p, p),
                   markerfacecolor="lightgray", markeredgecolor="black", markersize=8)
            for p in active_parts if p in PART_MARKERS_MPL
        ]

    ax.legend(handles=handles, loc="best", fontsize=fontsize)

print("Config complete \u2713")


In [ ]:
# === Cell 4: Filename-Derived Ground Truth ===
# Optional alternative to the GT CSV: the leading filename token is the polymer class
# (e.g. "HDPE_sample_01.csv"). INPUT_DIR is GT_PATH if it is a directory, otherwise
# BASE_DIR / DEFAULT_FOLDER (or BASE_DIR if that folder does not exist).

# --- Resolve INPUT_DIR ---
if GT_PATH.is_dir():
    INPUT_DIR = GT_PATH
else:
    _default_sub = BASE_DIR / DEFAULT_FOLDER
    INPUT_DIR = _default_sub if _default_sub.is_dir() else BASE_DIR
# INPUT_DIR = Path("/VOL_D/.../my_folder")   # <-- uncomment to override

# Leading filename token -> polymer class label.
# HDPE and LDPE stay separate classes (not merged into PE).
PREFIX_TO_CLASS = {
    "HDPE": "HDPE",
    "LDPE": "LDPE",
    "PET":  "PET",
    "PP":   "PP",
    "PS":   "PS",
}

# --- Derive GT ---
_PREFIX_RE = re.compile(r"^(?P<prefix>[A-Za-z0-9]+)_")

def derive_class_from_filename(name: str):
    m = _PREFIX_RE.match(name)
    if m is None:
        return None
    return PREFIX_TO_CLASS.get(m.group("prefix").upper())

_all_paths = sorted(p for p in INPUT_DIR.iterdir() if p.is_file() and not p.name.startswith("."))
_gt_lookup_new, _unmapped = {}, []
for _p in _all_paths:
    _cls = derive_class_from_filename(_p.name)
    if _cls is None:
        _unmapped.append(_p.name)
        continue
    _gt_lookup_new[_p.stem] = _cls

# --- Update global metadata ---
METADATA["gt_lookup"]   = _gt_lookup_new
METADATA["gt_source"]   = "filename"
METADATA["has_regions"] = False   # filename-derived data has no body/label/cap
METADATA["gt_csv"]      = None

_new_classes = sorted(set(_gt_lookup_new.values()))
CLASS_PALETTE = build_palette(_new_classes)

# --- Summary ---
print(f"INPUT_DIR          : {INPUT_DIR}")
print(f"Files discovered   : {len(_all_paths)}")
print(f"Mapped to a class  : {len(_gt_lookup_new)}")
print(f"Unmapped (excluded): {len(_unmapped)}")
if _unmapped:
    _preview = ", ".join(_unmapped[:10]) + (" …" if len(_unmapped) > 10 else "")
    print(f"  excluded names   : {_preview}")
print(f"\nClass distribution:")
_class_counts = pd.Series(list(_gt_lookup_new.values())).value_counts().sort_index()
print(_class_counts.to_string())
print(f"\nPalette: {CLASS_PALETTE}")
print(f"GT source set to: '{METADATA['gt_source']}', has_regions={METADATA['has_regions']}")

In [ ]:
# === Cell 5: Static PCA Viewer ===
# 2D or 3D PCA score plot (matplotlib) of the selected folder, coloured by GT class.

_subfolders_s = get_subfolders(BASE_DIR)
_default_s = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_s else (_subfolders_s[0] if _subfolders_s else "")

folder_widget_s = widgets.Dropdown(
    options=_subfolders_s, value=_default_s, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
view_widget_s = widgets.Dropdown(
    options=[("2D: PC1 vs PC2", "PC1_PC2"), ("2D: PC2 vs PC3", "PC2_PC3"),
             ("2D: PC1 vs PC3", "PC1_PC3"), ("3D: PC1 vs PC2 vs PC3", "3D")],
    value="PC1_PC2", description="View:",
)
body_widget_s       = widgets.Checkbox(value=True,  description="Body")
label_widget_s      = widgets.Checkbox(value=True,  description="Label")
cap_widget_s        = widgets.Checkbox(value=True,  description="Cap")
show_text_widget_s  = widgets.Checkbox(value=False, description="Show filenames")
save_button_s = widgets.Button(description="Save PNG", button_style="info", tooltip="Save current plot as PNG")

output_s        = widgets.Output()
status_output_s = widgets.Output()

_current_fig_s    = None
_current_folder_s = None
_current_view_s   = None


def draw_pca_s(*_args):
    global _current_fig_s, _current_folder_s, _current_view_s
    pc_index = {"PC1": 0, "PC2": 1, "PC3": 2}
    folder_name  = folder_widget_s.value
    view         = view_widget_s.value
    show_fnames  = show_text_widget_s.value
    has_regions  = METADATA["has_regions"]

    include_parts = None
    if has_regions:
        include_parts = [p for p, cb in
                         [("body", body_widget_s), ("label", label_widget_s), ("cap", cap_widget_s)]
                         if cb.value]

    with output_s:
        clear_output(wait=True)
        if not folder_name:
            print(f"No subfolders found in {BASE_DIR}")
            return
        try:
            plt.close("all")
            X_raw, meta, _ = load_spectra_matrix(BASE_DIR / folder_name, include_parts)
            X, plot_df = build_plot_df(X_raw, meta)
            _ensure_palette(plot_df["plot_class"])
            if plot_df.empty:
                print("No spectra found.")
                return

            pca = PCA(n_components=3)
            scores = pca.fit_transform(X)
            plot_df["PC1"], plot_df["PC2"], plot_df["PC3"] = scores[:, 0], scores[:, 1], scores[:, 2]

            if view == "3D":
                fig = plt.figure(figsize=(_FW, _FW * 0.6))
                ax  = fig.add_subplot(111, projection="3d")
            else:
                fig = plt.figure(figsize=(_FW * 0.6, _FW * 0.6))
                ax  = fig.add_subplot(111)
                ax.set_box_aspect(1)

            def _scatter(sub, marker, color):
                if view == "3D":
                    ax.scatter(sub["PC1"], sub["PC2"], sub["PC3"], s=55, marker=marker,
                               color=color, edgecolor="black", linewidth=0.4, alpha=0.9)
                else:
                    x_pc, y_pc = view.split("_")
                    ax.scatter(sub[x_pc], sub[y_pc], s=55, marker=marker,
                               color=color, edgecolor="black", linewidth=0.4, alpha=0.9)

            if has_regions:
                for part in sorted(plot_df["part"].dropna().unique()):
                    marker = PART_MARKERS_MPL.get(part, "o")
                    df_part = plot_df[plot_df["part"] == part]
                    for cls in sorted(df_part["plot_class"].unique()):
                        _scatter(df_part[df_part["plot_class"] == cls], marker,
                                 CLASS_PALETTE.get(cls, "#7f7f7f"))
            else:
                for cls in sorted(plot_df["plot_class"].unique()):
                    _scatter(plot_df[plot_df["plot_class"] == cls], "o",
                             CLASS_PALETTE.get(cls, "#7f7f7f"))

            if show_fnames:
                for _, row in plot_df.iterrows():
                    if view == "3D":
                        ax.text(row["PC1"], row["PC2"], row["PC3"], row["filename"], fontsize=_FONT_SIZE-3, alpha=0.8)
                    else:
                        x_pc, y_pc = view.split("_")
                        ax.annotate(row["filename"], (row[x_pc], row[y_pc]), fontsize=_FONT_SIZE-3, alpha=0.8)

            evr = pca.explained_variance_ratio_
            if view == "3D":
                ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
                ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
                ax.set_zlabel(f"PC3 ({evr[2]*100:.1f}%)")
            else:
                x_pc, y_pc = view.split("_")
                ax.set_xlabel(f"{x_pc} ({evr[pc_index[x_pc]]*100:.1f}%)")
                ax.set_ylabel(f"{y_pc} ({evr[pc_index[y_pc]]*100:.1f}%)")

            _leg_fs = (_FONT_SIZE + 1) if view == "3D" else (_FONT_SIZE - 1)
            build_mpl_legends(ax, plot_df, CLASS_PALETTE, show_parts=has_regions, fontsize=_leg_fs)
            plt.tight_layout()

            _current_fig_s, _current_folder_s, _current_view_s = fig, folder_name, view
            plt.show()
        except Exception as e:
            print(f"Error: {e}")


def on_save_s(b):
    with status_output_s:
        clear_output(wait=True)
        if _current_fig_s is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        png_path = SAVE_DIR / f"{PROJECT_NAME}_PCA_{_current_folder_s}_{_current_view_s}.png"
        _current_fig_s.savefig(png_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {png_path}")

save_button_s.on_click(on_save_s)

_watchers_s = [folder_widget_s, view_widget_s, show_text_widget_s]
if METADATA["has_regions"]:
    _watchers_s += [body_widget_s, label_widget_s, cap_widget_s]
for w in _watchers_s:
    w.observe(draw_pca_s, names="value")

if METADATA["has_regions"]:
    _rows_s = [widgets.HBox([folder_widget_s]),
               widgets.HBox([view_widget_s, body_widget_s, label_widget_s, cap_widget_s])]
else:
    _rows_s = [widgets.HBox([folder_widget_s]), widgets.HBox([view_widget_s])]

display(widgets.VBox(_rows_s + [widgets.HBox([show_text_widget_s, save_button_s]), status_output_s, output_s]))
draw_pca_s()

In [ ]:
# === Cell 6: Static t-SNE Viewer ===
# 2D t-SNE plot (matplotlib); perplexity is clamped to (n - 1) / 3 for small folders.

_subfolders_st = get_subfolders(BASE_DIR)
_default_st = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_st else (_subfolders_st[0] if _subfolders_st else "")

folder_widget_st = widgets.Dropdown(
    options=_subfolders_st, value=_default_st, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
body_widget_st       = widgets.Checkbox(value=True,  description="Body")
label_widget_st      = widgets.Checkbox(value=True,  description="Label")
cap_widget_st        = widgets.Checkbox(value=True,  description="Cap")
show_text_widget_st  = widgets.Checkbox(value=False, description="Show filenames")
perplexity_widget_st = widgets.IntSlider(value=30, min=5, max=50, step=1, description="Perplexity:",
                                         style={"description_width": "initial"}, continuous_update=False,
                                         layout=widgets.Layout(width="320px"))
pca_prereduce_widget_st = widgets.Checkbox(value=True, description="PCA pre-reduce (≤50D)")
save_button_st = widgets.Button(description="Save PNG", button_style="info", tooltip="Save current plot as PNG")

output_st        = widgets.Output()
status_output_st = widgets.Output()

_current_fig_st    = None
_current_folder_st = None
_current_label_st  = None


def draw_tsne2d_st(*_args):
    global _current_fig_st, _current_folder_st, _current_label_st
    folder_name = folder_widget_st.value
    show_fnames = show_text_widget_st.value
    perplexity  = perplexity_widget_st.value
    pca_pre     = pca_prereduce_widget_st.value
    has_regions = METADATA["has_regions"]

    include_parts = None
    if has_regions:
        include_parts = [p for p, cb in
                         [("body", body_widget_st), ("label", label_widget_st), ("cap", cap_widget_st)]
                         if cb.value]

    with output_st:
        clear_output(wait=True)
        if not folder_name:
            print(f"No subfolders found in {BASE_DIR}")
            return
        try:
            plt.close("all")
            X_raw, meta, _ = load_spectra_matrix(BASE_DIR / folder_name, include_parts)
            X, plot_df = build_plot_df(X_raw, meta)
            _ensure_palette(plot_df["plot_class"])
            if plot_df.empty:
                print("No spectra found.")
                return

            n_samples = X.shape[0]
            if n_samples < 3:
                print(f"Need at least 3 spectra for 2D t-SNE (got {n_samples}).")
                return
            eff_perp = max(2.0, min(float(perplexity), (n_samples - 1) / 3.0))

            X_in = X
            if pca_pre:
                n_comp = min(50, X.shape[1], n_samples - 1)
                if n_comp >= 2:
                    X_in = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit_transform(X)

            emb = TSNE(n_components=2, perplexity=eff_perp, init="pca",
                       learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X_in)
            plot_df["TSNE1"], plot_df["TSNE2"] = emb[:, 0], emb[:, 1]

            if eff_perp != float(perplexity):
                print(f"Note: perplexity clamped to {eff_perp:.0f} (requested {perplexity}, n={n_samples}).")

            fig, ax = plt.subplots(figsize=(_FW * 0.6, _FW * 0.6))
            ax.set_box_aspect(1)

            if has_regions:
                for part in sorted(plot_df["part"].dropna().unique()):
                    marker = PART_MARKERS_MPL.get(part, "o")
                    df_part = plot_df[plot_df["part"] == part]
                    for cls in sorted(df_part["plot_class"].unique()):
                        sub = df_part[df_part["plot_class"] == cls]
                        ax.scatter(sub["TSNE1"], sub["TSNE2"], s=55, marker=marker,
                                   color=CLASS_PALETTE.get(cls, "#7f7f7f"),
                                   edgecolor="black", linewidth=0.4, alpha=0.9)
            else:
                for cls in sorted(plot_df["plot_class"].unique()):
                    sub = plot_df[plot_df["plot_class"] == cls]
                    ax.scatter(sub["TSNE1"], sub["TSNE2"], s=55, marker="o",
                               color=CLASS_PALETTE.get(cls, "#7f7f7f"),
                               edgecolor="black", linewidth=0.4, alpha=0.9)

            if show_fnames:
                for _, row in plot_df.iterrows():
                    ax.annotate(row["filename"], (row["TSNE1"], row["TSNE2"]), fontsize=_FONT_SIZE-3, alpha=0.8)

            ax.set_xlabel("t-SNE 1")
            ax.set_ylabel("t-SNE 2")

            build_mpl_legends(ax, plot_df, CLASS_PALETTE, show_parts=has_regions)
            plt.tight_layout()

            _current_fig_st, _current_folder_st, _current_label_st = fig, folder_name, f"perp{eff_perp:.0f}"
            plt.show()
        except Exception as e:
            print(f"Error: {e}")


def on_save_st(b):
    with status_output_st:
        clear_output(wait=True)
        if _current_fig_st is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        png_path = SAVE_DIR / f"{PROJECT_NAME}_TSNE2D_{_current_folder_st}_{_current_label_st}.png"
        _current_fig_st.savefig(png_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {png_path}")

save_button_st.on_click(on_save_st)

_watchers_st = [folder_widget_st, show_text_widget_st, perplexity_widget_st, pca_prereduce_widget_st]
if METADATA["has_regions"]:
    _watchers_st += [body_widget_st, label_widget_st, cap_widget_st]
for w in _watchers_st:
    w.observe(draw_tsne2d_st, names="value")

_rows_st = [widgets.HBox([folder_widget_st])]
if METADATA["has_regions"]:
    _rows_st.append(widgets.HBox([body_widget_st, label_widget_st, cap_widget_st]))
_rows_st.append(widgets.HBox([perplexity_widget_st, pca_prereduce_widget_st]))
_rows_st.append(widgets.HBox([show_text_widget_st, save_button_st]))

display(widgets.VBox(_rows_st + [status_output_st, output_st]))
draw_tsne2d_st()

In [ ]:
# === Cell 7: Interactive 3D PCA Viewer ===
# Plotly figure rendered as inline HTML; clicking a point shows its filename.
# "Save PNG" writes a static render via kaleido; the toolbar camera icon saves the
# on-screen view.

_subfolders_dp = get_subfolders(BASE_DIR)
_default_dp = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_dp else (_subfolders_dp[0] if _subfolders_dp else "")

folder_widget_dp = widgets.Dropdown(
    options=_subfolders_dp, value=_default_dp, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
body_widget_dp       = widgets.Checkbox(value=True,  description="Body")
label_widget_dp      = widgets.Checkbox(value=True,  description="Label")
cap_widget_dp        = widgets.Checkbox(value=True,  description="Cap")
show_text_widget_dp  = widgets.Checkbox(value=False, description="Show filenames in plot")
save_button_dp = widgets.Button(description="Save PNG", button_style="info",
                                tooltip="Static PNG of current 3D figure (kaleido)")

output_dp        = widgets.Output()
status_output_dp = widgets.Output()

_current_fig_dp    = None   # Plotly Figure object (for saving)
_current_folder_dp = None


def _build_pca_fig(plot_df, pca, folder_name, show_filenames=False):
    fig = go.Figure()
    evr = pca.explained_variance_ratio_
    has_regions = METADATA["has_regions"]

    def _add(sub, name, symbol, color):
        if sub.empty:
            return
        cols = [sub["filename"].to_numpy(), sub["plot_class"].to_numpy()]
        if has_regions:
            cols.append(sub["part"].fillna("n/a").to_numpy())
        customdata = np.column_stack(cols)
        if has_regions:
            hov = ("<b>%{customdata[0]}</b><br>Class: %{customdata[1]}<br>Region: %{customdata[2]}<br>"
                   "PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>")
        else:
            hov = ("<b>%{customdata[0]}</b><br>Class: %{customdata[1]}<br>"
                   "PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>")
        fig.add_trace(go.Scatter3d(
            x=sub["PC1"], y=sub["PC2"], z=sub["PC3"],
            mode="markers+text" if show_filenames else "markers",
            text=sub["filename"] if show_filenames else None, textposition="top center",
            name=name, customdata=customdata, hovertemplate=hov,
            marker=dict(size=5, symbol=symbol, color=color, opacity=0.9,
                        line=dict(color="black", width=0.5)),
        ))

    if has_regions:
        for part in sorted(plot_df["part"].dropna().unique()):
            df_part = plot_df[plot_df["part"] == part]
            for cls in sorted(df_part["plot_class"].unique()):
                _add(df_part[df_part["plot_class"] == cls], f"{cls} {part}",
                     PART_SYMBOLS_PLOTLY.get(part, "circle"), CLASS_PALETTE.get(cls, "#7f7f7f"))
    else:
        for cls in sorted(plot_df["plot_class"].unique()):
            _add(plot_df[plot_df["plot_class"] == cls], cls, "circle", CLASS_PALETTE.get(cls, "#7f7f7f"))

    fig.update_layout(
        height=760, margin=dict(l=10, r=10, t=50, b=10),
        legend=dict(title="Class / Region" if has_regions else "Class", font=dict(size=14)),
        scene=dict(xaxis_title=f"PC1 ({evr[0]*100:.1f}%)",
                   yaxis_title=f"PC2 ({evr[1]*100:.1f}%)",
                   zaxis_title=f"PC3 ({evr[2]*100:.1f}%)", dragmode="turntable"),
    )
    return fig


def _fig_to_click_html(fig):
    div_id  = f"plotly_{uuid.uuid4().hex}"
    info_id = f"info_{uuid.uuid4().hex}"
    plot_html = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id=div_id,
                            config={"responsive": True, "displaylogo": False, "scrollZoom": True})
    return f"""
    <div id="{info_id}" style="margin:0 0 12px 0; padding:10px 12px;
        border:1px solid #d0d7de; border-radius:8px; background:#f6f8fa; font-family:sans-serif;">
        Click a point to see its filename.
    </div>
    {plot_html}
    <script>
    (function() {{
        const gd = document.getElementById("{div_id}");
        const info = document.getElementById("{info_id}");
        if (!gd || !info) return;
        function esc(v) {{
            return String(v ?? "").replace(/&/g,"&amp;").replace(/</g,"&lt;")
                .replace(/>/g,"&gt;").replace(/"/g,"&quot;").replace(/'/g,"&#39;");
        }}
        gd.on('plotly_click', function(data) {{
            if (!data || !data.points || !data.points.length) return;
            const pt = data.points[0], cd = pt.customdata || [];
            info.innerHTML =
                "<b>File:</b> " + esc(cd[0]) + "<br>" +
                "<b>Class:</b> " + esc(cd[1]) + "<br>" +
                "<b>PC1:</b> " + Number(pt.x).toFixed(3) + ", " +
                "<b>PC2:</b> " + Number(pt.y).toFixed(3) + ", " +
                "<b>PC3:</b> " + Number(pt.z).toFixed(3);
        }});
    }})();
    </script>
    """


def draw_pca_dp(*_args):
    global _current_fig_dp, _current_folder_dp
    folder_name = folder_widget_dp.value
    show_fnames = show_text_widget_dp.value
    has_regions = METADATA["has_regions"]

    include_parts = None
    if has_regions:
        include_parts = [p for p, cb in
                         [("body", body_widget_dp), ("label", label_widget_dp), ("cap", cap_widget_dp)]
                         if cb.value]

    with output_dp:
        clear_output(wait=True)
        if not folder_name:
            print(f"No subfolders found in {BASE_DIR}")
            return
        try:
            X_raw, meta, _ = load_spectra_matrix(BASE_DIR / folder_name, include_parts)
            X, plot_df = build_plot_df(X_raw, meta)
            _ensure_palette(plot_df["plot_class"])
            if plot_df.empty:
                print("No spectra found.")
                return

            pca = PCA(n_components=3)
            scores = pca.fit_transform(X)
            plot_df["PC1"], plot_df["PC2"], plot_df["PC3"] = scores[:, 0], scores[:, 1], scores[:, 2]

            fig = _build_pca_fig(plot_df, pca, folder_name, show_fnames)
            _current_fig_dp, _current_folder_dp = fig, folder_name

            clear_output(wait=True)
            display(HTML(_fig_to_click_html(fig)))
        except Exception as e:
            print(f"Error: {e}")


def on_save_dp(b):
    with status_output_dp:
        clear_output(wait=True)
        if _current_fig_dp is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        png_path = SAVE_DIR / f"{PROJECT_NAME}_PCA_{_current_folder_dp}_3D_{ts}.png"
        try:
            _current_fig_dp.write_image(str(png_path), width=1400, height=900, scale=2)
            print(f"Saved → {png_path}")
        except Exception as e:
            print(f"write_image failed ({e}). Tip: pip install kaleido")
            print("Alternatively, use the camera icon in the Plotly toolbar to download a PNG.")

save_button_dp.on_click(on_save_dp)

_watchers_dp = [folder_widget_dp, show_text_widget_dp]
if METADATA["has_regions"]:
    _watchers_dp += [body_widget_dp, label_widget_dp, cap_widget_dp]
for w in _watchers_dp:
    w.observe(draw_pca_dp, names="value")

_rows_dp = [widgets.HBox([folder_widget_dp])]
if METADATA["has_regions"]:
    _rows_dp.append(widgets.HBox([body_widget_dp, label_widget_dp, cap_widget_dp]))
_rows_dp.append(widgets.HBox([show_text_widget_dp, save_button_dp]))

display(widgets.VBox(_rows_dp + [status_output_dp, output_dp]))
draw_pca_dp()

In [ ]:
# === Cell 8: Interactive 3D t-SNE Viewer ===
# Plotly figure rendered as inline HTML. "Save PNG" writes a static render via
# kaleido; the toolbar camera icon saves the on-screen view.

_subfolders_t = get_subfolders(BASE_DIR)
_default_t = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_t else (_subfolders_t[0] if _subfolders_t else "")

folder_widget_t = widgets.Dropdown(
    options=_subfolders_t, value=_default_t, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
body_widget_t       = widgets.Checkbox(value=True,  description="Body")
label_widget_t      = widgets.Checkbox(value=True,  description="Label")
cap_widget_t        = widgets.Checkbox(value=True,  description="Cap")
show_text_widget_t  = widgets.Checkbox(value=False, description="Show filenames in plot")
perplexity_widget_t = widgets.IntSlider(value=30, min=5, max=50, step=1, description="Perplexity:",
                                        style={"description_width": "initial"}, continuous_update=False,
                                        layout=widgets.Layout(width="320px"))
pca_prereduce_widget_t = widgets.Checkbox(value=True, description="PCA pre-reduce (≤50D)")
save_button_t = widgets.Button(description="Save PNG", button_style="info",
                               tooltip="Static PNG of current 3D t-SNE (kaleido)")
recompute_button_t = widgets.Button(description="Recompute t-SNE", button_style="success",
                                    tooltip="Re-run t-SNE with the current settings")

output_t        = widgets.Output()
status_output_t = widgets.Output()

_current_fig_t    = None
_current_folder_t = None


def _build_tsne_fig(plot_df, folder_name, eff_perp, show_filenames=False):
    fig = go.Figure()
    has_regions = METADATA["has_regions"]

    def _add(sub, name, symbol, color):
        if sub.empty:
            return
        cols = [sub["filename"].to_numpy(), sub["plot_class"].to_numpy()]
        if has_regions:
            cols.append(sub["part"].fillna("n/a").to_numpy())
        customdata = np.column_stack(cols)
        if has_regions:
            hov = ("<b>%{customdata[0]}</b><br>Class: %{customdata[1]}<br>Region: %{customdata[2]}<br>"
                   "t-SNE1: %{x:.2f}<br>t-SNE2: %{y:.2f}<br>t-SNE3: %{z:.2f}<extra></extra>")
        else:
            hov = ("<b>%{customdata[0]}</b><br>Class: %{customdata[1]}<br>"
                   "t-SNE1: %{x:.2f}<br>t-SNE2: %{y:.2f}<br>t-SNE3: %{z:.2f}<extra></extra>")
        fig.add_trace(go.Scatter3d(
            x=sub["TSNE1"], y=sub["TSNE2"], z=sub["TSNE3"],
            mode="markers+text" if show_filenames else "markers",
            text=sub["filename"] if show_filenames else None, textposition="top center",
            name=name, customdata=customdata, hovertemplate=hov,
            marker=dict(size=5, symbol=symbol, color=color, opacity=0.9,
                        line=dict(color="black", width=0.5)),
        ))

    if has_regions:
        for part in sorted(plot_df["part"].dropna().unique()):
            df_part = plot_df[plot_df["part"] == part]
            for cls in sorted(df_part["plot_class"].unique()):
                _add(df_part[df_part["plot_class"] == cls], f"{cls} {part}",
                     PART_SYMBOLS_PLOTLY.get(part, "circle"), CLASS_PALETTE.get(cls, "#7f7f7f"))
    else:
        for cls in sorted(plot_df["plot_class"].unique()):
            _add(plot_df[plot_df["plot_class"] == cls], cls, "circle", CLASS_PALETTE.get(cls, "#7f7f7f"))

    fig.update_layout(
        height=760, margin=dict(l=10, r=10, t=50, b=10),
        legend=dict(title="Class / Region" if has_regions else "Class", font=dict(size=14)),
        scene=dict(xaxis_title="t-SNE 1", yaxis_title="t-SNE 2", zaxis_title="t-SNE 3",
                   dragmode="turntable"),
    )
    return fig


def _fig_to_click_html_t(fig):
    div_id  = f"plotly_{uuid.uuid4().hex}"
    info_id = f"info_{uuid.uuid4().hex}"
    plot_html = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id=div_id,
                            config={"responsive": True, "displaylogo": False, "scrollZoom": True})
    return f"""
    <div id="{info_id}" style="margin:0 0 12px 0; padding:10px 12px;
        border:1px solid #d0d7de; border-radius:8px; background:#f6f8fa; font-family:sans-serif;">
        Click a point to see its filename.
    </div>
    {plot_html}
    <script>
    (function() {{
        const gd = document.getElementById("{div_id}");
        const info = document.getElementById("{info_id}");
        if (!gd || !info) return;
        function esc(v) {{
            return String(v ?? "").replace(/&/g,"&amp;").replace(/</g,"&lt;")
                .replace(/>/g,"&gt;").replace(/"/g,"&quot;").replace(/'/g,"&#39;");
        }}
        gd.on('plotly_click', function(data) {{
            if (!data || !data.points || !data.points.length) return;
            const pt = data.points[0], cd = pt.customdata || [];
            info.innerHTML =
                "<b>File:</b> " + esc(cd[0]) + "<br>" +
                "<b>Class:</b> " + esc(cd[1]) + "<br>" +
                "<b>t-SNE1:</b> " + Number(pt.x).toFixed(3) + ", " +
                "<b>t-SNE2:</b> " + Number(pt.y).toFixed(3) + ", " +
                "<b>t-SNE3:</b> " + Number(pt.z).toFixed(3);
        }});
    }})();
    </script>
    """


def draw_tsne_3d(*_args):
    global _current_fig_t, _current_folder_t
    folder_name = folder_widget_t.value
    show_fnames = show_text_widget_t.value
    perplexity  = perplexity_widget_t.value
    pca_pre     = pca_prereduce_widget_t.value
    has_regions = METADATA["has_regions"]

    include_parts = None
    if has_regions:
        include_parts = [p for p, cb in
                         [("body", body_widget_t), ("label", label_widget_t), ("cap", cap_widget_t)]
                         if cb.value]

    with output_t:
        clear_output(wait=True)
        if not folder_name:
            print(f"No subfolders found in {BASE_DIR}")
            return
        try:
            print("Running t-SNE ...")
            X_raw, meta, _ = load_spectra_matrix(BASE_DIR / folder_name, include_parts)
            X, plot_df = build_plot_df(X_raw, meta)
            _ensure_palette(plot_df["plot_class"])
            if plot_df.empty:
                print("No spectra found.")
                return

            n_samples = X.shape[0]
            if n_samples < 4:
                print(f"Need at least 4 spectra for 3D t-SNE (got {n_samples}).")
                return
            eff_perp = max(2.0, min(float(perplexity), (n_samples - 1) / 3.0))

            X_in = X
            if pca_pre:
                n_comp = min(50, X.shape[1], n_samples - 1)
                if n_comp >= 2:
                    X_in = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit_transform(X)

            emb = TSNE(n_components=3, perplexity=eff_perp, init="pca",
                       learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X_in)
            plot_df["TSNE1"], plot_df["TSNE2"], plot_df["TSNE3"] = emb[:, 0], emb[:, 1], emb[:, 2]

            fig = _build_tsne_fig(plot_df, folder_name, eff_perp, show_fnames)
            _current_fig_t, _current_folder_t = fig, folder_name

            clear_output(wait=True)
            if eff_perp != float(perplexity):
                print(f"Note: perplexity clamped to {eff_perp:.0f} (requested {perplexity}, n={n_samples}).")
            display(HTML(_fig_to_click_html_t(fig)))
        except Exception as e:
            clear_output(wait=True)
            print(f"Error: {e}")


def on_save_t(b):
    with status_output_t:
        clear_output(wait=True)
        if _current_fig_t is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        png_path = SAVE_DIR / f"{PROJECT_NAME}_TSNE_{_current_folder_t}_3D_{ts}.png"
        try:
            _current_fig_t.write_image(str(png_path), width=1400, height=900, scale=2)
            print(f"Saved → {png_path}")
        except Exception as e:
            print(f"write_image failed ({e}). Tip: pip install kaleido")
            print("Alternatively, use the camera icon in the Plotly toolbar to download a PNG.")

save_button_t.on_click(on_save_t)
recompute_button_t.on_click(lambda b: draw_tsne_3d())

_watchers_t = [folder_widget_t, show_text_widget_t, perplexity_widget_t, pca_prereduce_widget_t]
if METADATA["has_regions"]:
    _watchers_t += [body_widget_t, label_widget_t, cap_widget_t]
for w in _watchers_t:
    w.observe(draw_tsne_3d, names="value")

_rows_t = [widgets.HBox([folder_widget_t])]
if METADATA["has_regions"]:
    _rows_t.append(widgets.HBox([body_widget_t, label_widget_t, cap_widget_t]))
_rows_t += [widgets.HBox([perplexity_widget_t, pca_prereduce_widget_t]),
            widgets.HBox([show_text_widget_t, save_button_t, recompute_button_t])]

display(widgets.VBox(_rows_t + [status_output_t, output_t]))
draw_tsne_3d()

In [ ]:
# === Cell 9: Static Spectra Viewer ===
# Plots selected raw spectra, overlaid or stacked. Labels read "GT class | filename";
# colours follow the class palette. Y-axis: reflectance.

_subfolders_sv = get_subfolders(BASE_DIR)
_default_sv = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_sv else (_subfolders_sv[0] if _subfolders_sv else "")

folder_widget_sv = widgets.Dropdown(
    options=_subfolders_sv, value=_default_sv, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
body_widget_sv  = widgets.Checkbox(value=True,  description="Body")
label_widget_sv = widgets.Checkbox(value=True,  description="Label")
cap_widget_sv   = widgets.Checkbox(value=True,  description="Cap")
file_selector_sv = widgets.SelectMultiple(options=[], value=(), description="Spectra:",
                                          style={"description_width": "initial"},
                                          layout=widgets.Layout(width="480px", height="180px"))
select_all_button_sv = widgets.Button(description="Select All", tooltip="Select all visible spectra",
                                      layout=widgets.Layout(width="110px"))
save_button_sv = widgets.Button(description="Save PNG", button_style="info", tooltip="Save current plot as PNG")
layout_widget_sv = widgets.ToggleButtons(
    options=[("Overlaid", "overlaid"), ("Stacked", "stacked")], value="overlaid",
    description="Layout:", style={"description_width": "initial"})
offset_widget_sv = widgets.FloatText(value=0.0, step=0.5, description="Stack offset (0=auto):",
                                     style={"description_width": "initial"}, layout=widgets.Layout(width="230px"))


def _auto_offset_sv(ys):
    spans = [np.percentile(y, 98) - np.percentile(y, 2) for y in ys if len(y)]
    return round(1.25 * float(np.median(spans)), 3) if spans else 1.0

output_sv        = widgets.Output()
status_output_sv = widgets.Output()

_current_fig_sv    = None
_current_folder_sv = None


def refresh_file_list_sv(*_args):
    folder_name = folder_widget_sv.value
    has_regions = METADATA["has_regions"]
    file_selector_sv.unobserve(draw_spectra_sv, names="value")

    if not folder_name or not (BASE_DIR / folder_name).is_dir():
        file_selector_sv.options = []
        file_selector_sv.value   = ()
        file_selector_sv.observe(draw_spectra_sv, names="value")
        draw_spectra_sv()
        return

    data_dir = BASE_DIR / folder_name
    if has_regions:
        include_parts = {p for p, cb in
                         [("body", body_widget_sv), ("label", label_widget_sv), ("cap", cap_widget_sv)]
                         if cb.value}
        files = sorted(f.name for f in data_dir.iterdir()
                       if f.is_file() and any(f.stem.lower().endswith(p) for p in include_parts))
    else:
        files = sorted(f.name for f in data_dir.iterdir()
                       if f.is_file() and not f.name.startswith("."))

    prev_sel = tuple(f for f in file_selector_sv.value if f in files)
    file_selector_sv.options = files
    file_selector_sv.value   = prev_sel
    file_selector_sv.observe(draw_spectra_sv, names="value")
    draw_spectra_sv()


def draw_spectra_sv(*_args):
    global _current_fig_sv, _current_folder_sv
    folder_name = folder_widget_sv.value
    selected    = list(file_selector_sv.value)

    with output_sv:
        clear_output(wait=True)
        if not folder_name:
            print("No source folder selected.")
            return
        if not selected:
            print("Select one or more spectra from the list above.")
            return

        data_dir = BASE_DIR / folder_name
        try:
            plt.close("all")
            fig, ax = plt.subplots(figsize=(_FW, _FW * 0.5))

            _series_sv = []
            for fname in selected:
                stem = Path(fname).stem
                gt_class = get_gt_class(stem)
                if gt_class is None:
                    base_name, part = parse_filename(fname)
                    gt_class = lookup_gt_class(base_name, part)
                class_label = gt_class if gt_class else "unknown"
                _ensure_palette([class_label])
                color = CLASS_PALETTE.get(class_label, "#7f7f7f")

                df = pd.read_csv(data_dir / fname, sep=";", header=None, names=["Wavenumber", "Reflectance"])
                df["Wavenumber"]  = pd.to_numeric(df["Wavenumber"], errors="coerce")
                df["Reflectance"] = pd.to_numeric(df["Reflectance"], errors="coerce")
                df = df.dropna().sort_values("Wavenumber")
                _series_sv.append((df["Wavenumber"].to_numpy(float),
                                   df["Reflectance"].to_numpy(float), class_label, color, fname))

            if layout_widget_sv.value == "stacked" and _series_sv:
                _off_sv = (offset_widget_sv.value if offset_widget_sv.value > 0
                           else _auto_offset_sv([s[1] for s in _series_sv]))
                for _i, (xw, yr, class_label, color, fname) in enumerate(_series_sv):
                    _base = _i * _off_sv
                    ax.plot(xw, yr + _base, color=color, linewidth=1.5, alpha=0.92)
                    ax.text(0.006, _base, f"{class_label} | {fname}", color=color,
                            fontsize=_FONT_SIZE - 2, va="center", ha="left",
                            transform=ax.get_yaxis_transform())
                ax.set_yticks([])
            else:
                for xw, yr, class_label, color, fname in _series_sv:
                    ax.plot(xw, yr, color=color, linewidth=1.5, alpha=0.92,
                            label=f"{class_label} | {fname}")
                ax.legend(loc="best", fontsize=_FONT_SIZE-1)

            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Reflectance")
            ax.invert_xaxis()
            ax.grid(True, alpha=0.25)
            plt.tight_layout()

            _current_fig_sv, _current_folder_sv = fig, folder_name
            plt.show()
        except Exception as e:
            print(f"Error: {e}")


def on_select_all_sv(b):
    file_selector_sv.value = tuple(file_selector_sv.options)


def on_save_sv(b):
    with status_output_sv:
        clear_output(wait=True)
        if _current_fig_sv is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        n = len(file_selector_sv.value)
        png_path = SAVE_DIR / f"{PROJECT_NAME}_Spectra_{_current_folder_sv}_{n}sel.png"
        _current_fig_sv.savefig(png_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {png_path}")

select_all_button_sv.on_click(on_select_all_sv)
save_button_sv.on_click(on_save_sv)

folder_widget_sv.observe(refresh_file_list_sv, names="value")
if METADATA["has_regions"]:
    body_widget_sv.observe(refresh_file_list_sv, names="value")
    label_widget_sv.observe(refresh_file_list_sv, names="value")
    cap_widget_sv.observe(refresh_file_list_sv, names="value")
file_selector_sv.observe(draw_spectra_sv, names="value")
layout_widget_sv.observe(draw_spectra_sv, names="value")
offset_widget_sv.observe(draw_spectra_sv, names="value")

_rows_sv = [widgets.HBox([folder_widget_sv])]
if METADATA["has_regions"]:
    _rows_sv.append(widgets.HBox([body_widget_sv, label_widget_sv, cap_widget_sv]))
_rows_sv += [widgets.HBox([file_selector_sv, widgets.VBox([select_all_button_sv])]),
             widgets.HBox([layout_widget_sv, offset_widget_sv]),
             widgets.HBox([save_button_sv])]

display(widgets.VBox(_rows_sv + [status_output_sv, output_sv]))
refresh_file_list_sv()

In [ ]:
# === Cell 10: Group Spectra View ===
# Two figures: wavelength (nm) and wavenumber (1/cm). Single View (individual spectra)
# or Aggregated Group View (mean ± std), with multi-select filters. Without
# has_regions, the Region filter and "by region" grouping are disabled.

import random as _gs_random

display(HTML("""
<style>
.gs-title { font-weight: 700; color: #1f2937; margin: 0 0 6px 0; font-size: 16px; }
.gs-subtitle { color: #4b5563; margin: 0 0 10px 0; font-size: 12px; }
.gs-status { padding: 6px 10px; border-radius: 6px; border: 1px solid #d1d5db;
             background: #f8fafc; color: #111827; font-size: 12px; }
</style>
"""))

_GS_STYLE = {"description_width": "initial"}
_GS_FNAME_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_number>\d+)_(?P<region>[A-Za-z0-9]+)$")
_GS_SIMPLE_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_")

gs_last_figure = {"fig": None, "path": None}


def _gs_discover_folders():
    options = []
    if BASE_DIR.exists():
        for sf in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()], key=lambda p: p.name):
            options.append((sf.name, str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options


def _gs_read_spectrum(path):
    from io import StringIO
    path = Path(path)
    raw_text = path.read_text(encoding="utf-8", errors="replace")
    lines = raw_text.strip().splitlines()
    data_lines = [l for l in lines if not l.startswith("##")]
    if not data_lines:
        raise ValueError("File is empty after stripping headers.")
    block = "\n".join(data_lines)
    sep = ";" if ";" in data_lines[0] else r"\s+"
    df = pd.read_csv(StringIO(block), sep=sep, header=None,
                     names=["wavenumber", "signal"], engine="python")
    df["wavenumber"] = pd.to_numeric(df["wavenumber"], errors="coerce")
    df["signal"] = pd.to_numeric(df["signal"], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    if df.empty:
        raise ValueError("File contains no valid numeric rows.")
    return df


def _gs_folder_metadata(folder):
    folder = Path(folder)
    rows = []
    has_regions = METADATA["has_regions"]
    gt_lookup = METADATA["gt_lookup"]
    if not folder.exists():
        return pd.DataFrame(columns=["filename", "material", "object_number", "region", "path", "gt_class"])
    for p in sorted([f for f in folder.iterdir() if f.is_file() and not f.name.startswith(".")],
                    key=lambda p: p.name):
        stem = p.stem
        if has_regions:
            m = _GS_FNAME_RE.match(stem)
            if m is None:
                continue
            info = m.groupdict()
            gt = gt_lookup.get(stem)
            if gt is None:
                base_name = f"{info['material']}_obj_{info['object_number']}"
                gt = lookup_gt_class(base_name, info["region"])
            rows.append({"filename": stem, "material": info["material"],
                         "object_number": info["object_number"], "region": info["region"],
                         "path": str(p), "gt_class": gt if gt else "unknown"})
        else:
            m = _GS_SIMPLE_RE.match(stem)
            material = m.group("material") if m else "unknown"
            rows.append({"filename": stem, "material": material, "object_number": "0",
                         "region": "n/a", "path": str(p),
                         "gt_class": gt_lookup.get(stem, "unknown")})
    return pd.DataFrame(rows)


def _gs_filter(df, materials, regions, objects, filename, gt_classes):
    out = df
    def _apply(o, col, sel):
        if sel:  # non-empty tuple = active filter
            return o[o[col].astype(str).isin([str(x) for x in sel])]
        return o
    out = _apply(out, "material", materials)
    out = _apply(out, "region", regions)
    out = _apply(out, "object_number", objects)
    out = _apply(out, "gt_class", gt_classes)
    if filename != "All":
        out = out[out["filename"] == filename]
    return out.reset_index(drop=True)


def _gs_align(dataframes):
    ref_x = dataframes[0]["wavenumber"].to_numpy(dtype=float)
    aligned = []
    for df in dataframes:
        x = df["wavenumber"].to_numpy(dtype=float)
        y = df["signal"].to_numpy(dtype=float)
        if len(x) == len(ref_x) and np.allclose(x, ref_x):
            aligned.append(y)
            continue
        descending = np.any(np.diff(x) < 0)
        x_asc = x[::-1] if descending else x
        y_asc = y[::-1] if descending else y
        ref_desc = np.any(np.diff(ref_x) < 0)
        ref_asc = ref_x[::-1] if ref_desc else ref_x
        interp_y = np.interp(ref_asc, x_asc, y_asc)
        aligned.append(interp_y[::-1] if ref_desc else interp_y)
    return ref_x, np.vstack(aligned)


def _gs_wn_to_wl(wavenumber_values):
    wn = np.asarray(wavenumber_values, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(wn != 0, 1e7 / wn, np.nan)


def _gs_sort_wl(x_wavenumber, y_values):
    wl = _gs_wn_to_wl(x_wavenumber)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(wl) & np.isfinite(y)
    wl, y = wl[mask], y[mask]
    order = np.argsort(wl)
    return wl[order], y[order]


def _gs_color(label, idx, cmap):
    """Class-palette colour when the group label is a known class, else cmap fallback."""
    c = CLASS_PALETTE.get(str(label))
    return c if c is not None else cmap(idx)


def _gs_style_axis(ax, domain):
    """domain: 'wavenumber' or 'wavelength'."""
    if domain == "wavenumber":
        ax.set_xlabel("Wavenumber (1/cm)")
        ax.set_ylabel("Reflectance")
        ax.grid(True, alpha=0.25)
        ax.invert_xaxis()
    else:
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Reflectance")
        ax.grid(True, alpha=0.25)


def _gs_auto_offset(y_list):
    spans = [np.percentile(y, 98) - np.percentile(y, 2) for y in y_list if len(y)]
    return round(1.25 * float(np.median(spans)), 3) if spans else 1.0


def _gs_stacked_label(ax, base, text, color):
    ax.text(0.006, base, text, color=color, fontsize=_FONT_SIZE - 1,
            va="center", ha="left", fontweight="bold",
            transform=ax.get_yaxis_transform())


def _gs_build_groups(df, grouping_mode):
    groups = {}
    if grouping_mode == "by material":
        for name, gdf in df.groupby("material"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by region":
        for name, gdf in df.groupby("region"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by object":
        for name, gdf in df.groupby("object_number"):
            groups[f"obj_{name}"] = gdf.reset_index(drop=True)
    elif grouping_mode == "by GT class":
        col = "gt_class" if "gt_class" in df.columns else "material"
        for name, gdf in df.groupby(col):
            groups[str(name)] = gdf.reset_index(drop=True)
    else:
        groups["all"] = df.reset_index(drop=True)
    return groups


def _gs_plot_single_domain(ax, plot_records, domain, layout="overlaid", offset=0.0):
    cmap = plt.get_cmap("tab20", max(len(plot_records), 1))
    prepared = []
    for idx, record in enumerate(plot_records):
        df = record["df"]
        wn = df["wavenumber"].to_numpy(dtype=float)
        y = df["signal"].to_numpy(dtype=float)
        if domain == "wavenumber":
            x, y_plot = wn, y
        else:
            x, y_plot = _gs_sort_wl(wn, y)
        gt_class = record.get("gt_class", "unknown")
        color = _gs_color(gt_class, idx, cmap)
        label = f'{gt_class} | {record["filename"]}'
        prepared.append((x, y_plot, color, label))
    if layout == "stacked" and prepared:
        off = offset if offset > 0 else _gs_auto_offset([p[1] for p in prepared])
        for i, (x, y_plot, color, label) in enumerate(prepared):
            base = i * off
            ax.plot(x, y_plot + base, color=color, linewidth=1.5, alpha=0.95)
            _gs_stacked_label(ax, base, label, color)
        ax.set_yticks([])
    else:
        for x, y_plot, color, label in prepared:
            ax.plot(x, y_plot, color=color, linewidth=1.5, alpha=0.95, label=label)
        ax.legend(loc="best", fontsize=_FONT_SIZE-1, ncol=1)


def _gs_plot_group_domain(ax, aggregated, domain, layout="overlaid", offset=0.0):
    cmap = plt.get_cmap("tab10", max(len(aggregated), 1))
    prepared = []
    for idx, (legend_label, spectra_list) in enumerate(aggregated.items()):
        ref_x, stack = _gs_align(spectra_list)
        mean_y = stack.mean(axis=0)
        std_y = stack.std(axis=0, ddof=0)
        if domain == "wavenumber":
            x, m, s = ref_x, mean_y, std_y
        else:
            x, m = _gs_sort_wl(ref_x, mean_y)
            _, s = _gs_sort_wl(ref_x, std_y)
        color = _gs_color(legend_label, idx, cmap)
        prepared.append((x, m, s, color, legend_label))
    if layout == "stacked" and prepared:
        off = offset if offset > 0 else _gs_auto_offset([p[1] for p in prepared])
        for i, (x, m, s, color, legend_label) in enumerate(prepared):
            base = i * off
            ax.plot(x, m + base, color=color, linewidth=2.0)
            ax.fill_between(x, m + base - s, m + base + s, color=color, alpha=0.18)
            _gs_stacked_label(ax, base, legend_label, color)
        ax.set_yticks([])
    else:
        for x, m, s, color, legend_label in prepared:
            ax.plot(x, m, color=color, linewidth=2.0, label=legend_label)
            ax.fill_between(x, m - s, m + s, color=color, alpha=0.18)
        ax.legend(loc="best", fontsize=_FONT_SIZE-1)


def gs_save_plot(_):
    fig_wn = gs_last_figure.get("fig_wn")
    fig_wl = gs_last_figure.get("fig_wl")
    if fig_wn is None and fig_wl is None:
        gs_status.value = '<div class="gs-status">No plot is currently available to save.</div>'
        return
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    saved_paths = []
    try:
        if fig_wn is not None:
            p = SAVE_DIR / f"{PROJECT_NAME}_GroupSpectra_Wavenumber_{timestamp}.png"
            fig_wn.savefig(p, dpi=150, bbox_inches="tight")
            saved_paths.append(str(p))
        if fig_wl is not None:
            p = SAVE_DIR / f"{PROJECT_NAME}_GroupSpectra_Wavelength_{timestamp}.png"
            fig_wl.savefig(p, dpi=150, bbox_inches="tight")
            saved_paths.append(str(p))
        gs_last_figure["path"] = saved_paths
        gs_status.value = ('<div class="gs-status">Plots saved:<br>'
                           + "<br>".join(f"<b>{p}</b>" for p in saved_paths) + '</div>')
    except Exception as exc:
        gs_status.value = f'<div class="gs-status">Failed to save plot(s): {exc}</div>'


def gs_update(_=None):
    gs_plot_output.clear_output(wait=True)
    gs_last_figure["fig_wn"] = None
    gs_last_figure["fig_wl"] = None
    plt.close("all")   # close any open figures first

    folder_path = gs_source_dropdown.value
    if folder_path is None:
        gs_status.value = '<div class="gs-status">No source folder selected.</div>'
        return

    folder_name = Path(folder_path).name
    meta_df = _gs_folder_metadata(folder_path)
    if meta_df.empty:
        gs_status.value = f'<div class="gs-status">No parseable spectra found in {folder_name}.</div>'
        return

    catalog = _gs_filter(meta_df, materials=gs_material.value, regions=gs_region.value,
                         objects=gs_object.value, filename=gs_filename.value,
                         gt_classes=gs_gt_class.value)
    if catalog.empty:
        gs_status.value = '<div class="gs-status">No spectra match the selected filters.</div>'
        return

    _ensure_palette(catalog["gt_class"].unique())

    # --- Step 1: gather data; no figure is created before data is available ---
    warnings = []
    if gs_mode.value == "single":
        plot_records = []
        for _, row in catalog.iterrows():
            try:
                plot_records.append({"filename": row["filename"],
                                     "df": _gs_read_spectrum(row["path"]),
                                     "gt_class": row.get("gt_class", "unknown")})
            except Exception as exc:
                warnings.append(f'{Path(row["path"]).name} failed ({exc})')
        if not plot_records:
            gs_status.value = '<div class="gs-status">No spectra were readable for Single View.</div>'
            return
        n_series = len(plot_records)
        mode_text = "Single View"
    else:
        groups = _gs_build_groups(catalog, gs_grouping.value)
        aggregated = {}
        for legend_label, group_df in groups.items():
            spectra_list = []
            for _, row in group_df.iterrows():
                try:
                    spectra_list.append(_gs_read_spectrum(row["path"]))
                except Exception as exc:
                    warnings.append(f'{Path(row["path"]).name} failed ({exc})')
            if spectra_list:
                aggregated[legend_label] = spectra_list
        if not aggregated:
            gs_status.value = '<div class="gs-status">No grouped spectra were readable.</div>'
            return
        n_series = len(aggregated)
        mode_text = f"Aggregated Group View ({gs_grouping.value})"

    # --- Step 2: build and display two separate figures ---
    try:
        with gs_plot_output:
            clear_output(wait=True)

            fig_wn, ax_wn = plt.subplots(figsize=(_FW, _FW * 0.5), constrained_layout=True)
            if gs_mode.value == "single":
                _gs_plot_single_domain(ax_wn, plot_records, "wavenumber", layout=gs_layout.value, offset=gs_offset.value)
            else:
                _gs_plot_group_domain(ax_wn, aggregated, "wavenumber", layout=gs_layout.value, offset=gs_offset.value)
            _gs_style_axis(ax_wn, "wavenumber")
            display(fig_wn)
            plt.close(fig_wn)   # deregister from pyplot state, object reference kept below

            fig_wl, ax_wl = plt.subplots(figsize=(_FW, _FW * 0.5), constrained_layout=True)
            if gs_mode.value == "single":
                _gs_plot_single_domain(ax_wl, plot_records, "wavelength", layout=gs_layout.value, offset=gs_offset.value)
            else:
                _gs_plot_group_domain(ax_wl, aggregated, "wavelength", layout=gs_layout.value, offset=gs_offset.value)
            _gs_style_axis(ax_wl, "wavelength")
            display(fig_wl)
            plt.close(fig_wl)

        gs_last_figure["fig_wn"] = fig_wn
        gs_last_figure["fig_wl"] = fig_wl

        warning_text = "" if not warnings else " Warnings: " + "; ".join(warnings[:8])
        gs_status.value = (f'<div class="gs-status">{mode_text} updated with <b>{n_series}</b> series '
                           f'from {folder_name}.{warning_text}</div>')
    except Exception as exc:
        gs_last_figure["fig_wn"] = None
        gs_last_figure["fig_wl"] = None
        gs_status.value = f'<div class="gs-status">Visualization update failed: {exc}</div>'


def gs_refresh_filters(_=None):
    folder_path = gs_source_dropdown.value
    has_regions = METADATA["has_regions"]
    if folder_path is None:
        for w in (gs_material, gs_region, gs_gt_class, gs_object):
            w.options = []
            w.value = ()
        gs_filename.options = ["All"]
        gs_filename.value = "All"
        return

    meta_df = _gs_folder_metadata(folder_path)
    if meta_df.empty:
        for w in (gs_material, gs_region, gs_gt_class, gs_object):
            w.options = []
            w.value = ()
        gs_filename.options = ["All"]
        gs_filename.value = "All"
        gs_update()
        return

    materials  = sorted(meta_df["material"].dropna().astype(str).unique().tolist())
    gt_classes = sorted(meta_df["gt_class"].dropna().astype(str).unique().tolist())
    objects    = sorted(meta_df["object_number"].dropna().astype(str).unique().tolist())

    gs_material.options = materials
    gs_material.value = tuple(v for v in gs_material.value if v in materials)

    if has_regions:
        regions = sorted(meta_df["region"].dropna().astype(str).unique().tolist())
        gs_region.options = regions
        gs_region.disabled = False
        gs_region.value = tuple(v for v in gs_region.value if v in regions)
    else:
        gs_region.options = []
        gs_region.value = ()
        gs_region.disabled = True

    gs_object.options = objects
    gs_object.value = tuple(v for v in gs_object.value if v in objects)

    gs_gt_class.options = gt_classes
    gs_gt_class.value = tuple(v for v in gs_gt_class.value if v in gt_classes)

    filtered = _gs_filter(meta_df, materials=gs_material.value, regions=gs_region.value,
                          objects=gs_object.value, filename="All", gt_classes=gs_gt_class.value)
    filenames = ["All"] + sorted(filtered["filename"].dropna().astype(str).unique().tolist())
    gs_filename.options = filenames
    if gs_filename.value not in filenames:
        gs_filename.value = "All"

    if has_regions:
        gs_grouping.options = ["all", "by material", "by region", "by object", "by GT class"]
    else:
        gs_grouping.options = ["all", "by material", "by object", "by GT class"]
        if gs_grouping.value == "by region":
            gs_grouping.value = "by material"

    gs_update()


def gs_random_file(_):
    options = [opt for opt in gs_filename.options if opt != "All"]
    if options:
        gs_filename.value = _gs_random.choice(options)


def gs_refresh_folders(_=None):
    gs_source_dropdown.options = _gs_discover_folders()
    gs_refresh_filters()


# --- Widgets ---
_gs_folder_options = _gs_discover_folders()
gs_source_dropdown = widgets.Dropdown(
    options=_gs_folder_options,
    value=_gs_folder_options[0][1] if _gs_folder_options else None,
    description="Source folder", style=_GS_STYLE, layout=widgets.Layout(width="500px"))

_gs_sel_layout = widgets.Layout(width="200px", height="120px")
gs_material = widgets.SelectMultiple(options=[], value=(), description="Material", style=_GS_STYLE, layout=_gs_sel_layout)
gs_region   = widgets.SelectMultiple(options=[], value=(), description="Region", style=_GS_STYLE, layout=_gs_sel_layout,
                                     disabled=not METADATA["has_regions"])
gs_gt_class = widgets.SelectMultiple(options=[], value=(), description="GT class", style=_GS_STYLE, layout=_gs_sel_layout)
gs_object   = widgets.SelectMultiple(options=[], value=(), description="Object number", style=_GS_STYLE, layout=_gs_sel_layout)
gs_filename = widgets.Dropdown(options=["All"], value="All", description="Specific filename", style=_GS_STYLE, layout=widgets.Layout(width="340px"))
gs_mode = widgets.ToggleButtons(options=[("Single View", "single"), ("Aggregated Group View", "aggregated")], value="single", description="Mode", style=_GS_STYLE)

_gs_grouping_opts = ["all", "by material", "by region", "by object", "by GT class"] if METADATA["has_regions"] else ["all", "by material", "by object", "by GT class"]
gs_grouping = widgets.Dropdown(options=_gs_grouping_opts, value="by material", description="Group by", style=_GS_STYLE, layout=widgets.Layout(width="220px"))
gs_layout = widgets.ToggleButtons(options=[("Overlaid", "overlaid"), ("Stacked", "stacked")],
                                  value="overlaid", description="Layout", style=_GS_STYLE)
gs_offset = widgets.FloatText(value=0.0, step=0.5, description="Stack offset (0=auto)",
                              style=_GS_STYLE, layout=widgets.Layout(width="240px"))

gs_random_button  = widgets.Button(description="Random", icon="random", layout=widgets.Layout(width="150px", height="34px"))
gs_refresh_button = widgets.Button(description="Refresh folders", icon="refresh", button_style="info", layout=widgets.Layout(width="150px", height="34px"))
gs_save_button    = widgets.Button(description="Save Plot", icon="save", button_style="success", layout=widgets.Layout(width="150px", height="34px"))
gs_status = widgets.HTML('<div class="gs-status">Select a source folder to inspect spectra.</div>')
gs_plot_output = widgets.Output(layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="8px"))

gs_source_dropdown.observe(gs_refresh_filters, names="value")
gs_refresh_button.on_click(gs_refresh_folders)
gs_random_button.on_click(gs_random_file)
gs_save_button.on_click(gs_save_plot)
for w in (gs_material, gs_region, gs_object, gs_gt_class):
    w.observe(gs_refresh_filters, names="value")
for w in (gs_filename, gs_mode, gs_grouping, gs_layout, gs_offset):
    w.observe(gs_update, names="value")

gs_controls = widgets.VBox([
    widgets.HBox([gs_source_dropdown, gs_refresh_button]),
    widgets.HBox([gs_material, gs_region, gs_gt_class, gs_object]),
    widgets.HBox([gs_filename, gs_random_button]),
    widgets.HBox([gs_mode, gs_grouping]),
    widgets.HBox([gs_layout, gs_offset]),
    widgets.HBox([gs_save_button]),
])

gs_panel = widgets.VBox([
    widgets.HTML(
        '<div class="gs-title">Group Spectra View</div>'
        '<div class="gs-subtitle">Select any subfolder from the dataset. Filter by filename-derived '
        'material, region (if available), object, or ground-truth class (multi-select). Single View shows '
        'individual spectra; Aggregated View shows mean ± std curves grouped by your choice. Two separate '
        'plots are produced and saved: one on the wavenumber (1/cm) axis, one on the wavelength (nm) '
        'axis.</div>'
    ),
    gs_controls, gs_status, gs_plot_output,
], layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="12px", margin="8px 0"))

display(gs_panel)
gs_refresh_filters()

In [ ]:
# === Cell 11: Peak Finder ===
# Previews scipy find_peaks (valleys or maxima) on one spectrum; "Run on all" exports
# a per-file peak table (ranked by prominence) and a parameter log to SAVE_DIR.

from scipy.signal import find_peaks, peak_prominences

peak_direction_pk = widgets.ToggleButtons(
    options=[("Valleys (local minima)", "valleys"), ("Maxima (local maxima)", "maxima")],
    value="valleys", description="Peak direction:",
    style={"description_width": "initial", "button_width": "180px"})

_subfolders_pk = get_subfolders(BASE_DIR)
_default_pk = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_pk else (_subfolders_pk[0] if _subfolders_pk else "")

folder_widget_pk = widgets.Dropdown(
    options=_subfolders_pk, value=_default_pk, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"))
body_widget_pk  = widgets.Checkbox(value=True, description="Body")
label_widget_pk = widgets.Checkbox(value=True, description="Label")
cap_widget_pk   = widgets.Checkbox(value=True, description="Cap")
file_widget_pk = widgets.Dropdown(options=[], value=None, description="Preview file:",
                                  style={"description_width": "initial"}, layout=widgets.Layout(width="480px"))
random_button_pk = widgets.Button(description="Random", icon="random", layout=widgets.Layout(width="110px"))
prominence_pk = widgets.FloatText(value=0.05, step=0.01, description="Min prominence:",
                                  style={"description_width": "initial"}, layout=widgets.Layout(width="230px"))
height_pk     = widgets.FloatText(value=0.0, step=0.01, description="Min height:",
                                  style={"description_width": "initial"}, layout=widgets.Layout(width="230px"))
distance_pk   = widgets.BoundedIntText(value=1, min=1, max=100000, description="Min distance (pts):",
                                       style={"description_width": "initial"}, layout=widgets.Layout(width="230px"))
width_pk      = widgets.FloatText(value=0.0, step=0.5, description="Min width (pts):",
                                  style={"description_width": "initial"}, layout=widgets.Layout(width="230px"))
run_button_pk = widgets.Button(description="Run on all + export CSV", button_style="success", icon="play",
                               layout=widgets.Layout(width="220px"))

output_pk        = widgets.Output()
status_output_pk = widgets.Output()

_current_fig_pk    = None
_current_folder_pk = None


def _read_xy_pk(path):
    df = pd.read_csv(path, sep=";", header=None, names=["Wavenumber", "Reflectance"])
    df["Wavenumber"]  = pd.to_numeric(df["Wavenumber"], errors="coerce")
    df["Reflectance"] = pd.to_numeric(df["Reflectance"], errors="coerce")
    df = df.dropna().sort_values("Wavenumber")
    return df["Wavenumber"].to_numpy(), df["Reflectance"].to_numpy()


def _peak_kwargs_pk():
    kwargs = {}
    if prominence_pk.value and prominence_pk.value > 0:
        kwargs["prominence"] = prominence_pk.value
    if height_pk.value and height_pk.value > 0:
        kwargs["height"] = height_pk.value
    if distance_pk.value and distance_pk.value > 1:
        kwargs["distance"] = int(distance_pk.value)
    if width_pk.value and width_pk.value > 0:
        kwargs["width"] = width_pk.value
    return kwargs


def _detect_peaks_pk(wn, y, direction="valleys"):
    y_detect = -y if direction == "valleys" else y
    peaks, _props = find_peaks(y_detect, **_peak_kwargs_pk())
    if len(peaks) == 0:
        return np.array([], dtype=int), np.array([])
    proms = peak_prominences(y_detect, peaks)[0]
    order = np.argsort(proms)[::-1]
    return peaks[order], proms[order]


def refresh_file_list_pk(*_args):
    folder_name = folder_widget_pk.value
    has_regions = METADATA["has_regions"]
    file_widget_pk.unobserve(draw_peaks_pk, names="value")

    if not folder_name or not (BASE_DIR / folder_name).is_dir():
        file_widget_pk.options = []
        file_widget_pk.value   = None
        file_widget_pk.observe(draw_peaks_pk, names="value")
        draw_peaks_pk()
        return

    data_dir = BASE_DIR / folder_name
    if has_regions:
        include_parts = {p for p, cb in
                         [("body", body_widget_pk), ("label", label_widget_pk), ("cap", cap_widget_pk)]
                         if cb.value}
        files = sorted(f.name for f in data_dir.iterdir()
                       if f.is_file() and any(f.stem.lower().endswith(p) for p in include_parts))
    else:
        files = sorted(f.name for f in data_dir.iterdir()
                       if f.is_file() and not f.name.startswith("."))

    prev = file_widget_pk.value if file_widget_pk.value in files else (files[0] if files else None)
    file_widget_pk.options = files
    file_widget_pk.value   = prev
    file_widget_pk.observe(draw_peaks_pk, names="value")
    draw_peaks_pk()


def draw_peaks_pk(*_args):
    global _current_fig_pk, _current_folder_pk
    folder_name = folder_widget_pk.value
    fname       = file_widget_pk.value
    direction   = peak_direction_pk.value

    with output_pk:
        clear_output(wait=True)
        if not folder_name:
            print("No source folder selected.")
            return
        if not fname:
            print("No file selected (check the part filters and folder).")
            return
        try:
            plt.close("all")
            wn, y = _read_xy_pk(BASE_DIR / folder_name / fname)
            peaks, proms = _detect_peaks_pk(wn, y, direction=direction)

            fig, ax = plt.subplots(figsize=(_FW, _FW * 0.5))
            stem = Path(fname).stem
            gt_class = get_gt_class(stem)
            if gt_class is None:
                base_name, part = parse_filename(fname)
                gt_class = lookup_gt_class(base_name, part)
            gt_class = gt_class or "unknown"
            _ensure_palette([gt_class])
            color = CLASS_PALETTE.get(gt_class, "#7f7f7f")

            ax.plot(wn, y, color=color, linewidth=1.5, alpha=0.9, label=f"{gt_class} | {fname}")

            direction_label = "valleys" if direction == "valleys" else "peaks"
            marker_shape = "v" if direction == "valleys" else "^"
            if len(peaks) > 0:
                ax.scatter(wn[peaks], y[peaks], color="#d62728", s=40, zorder=5,
                           marker=marker_shape, label=f"{len(peaks)} {direction_label}")
                for pk in peaks:
                    ax.axvline(wn[pk], color="#d62728", linewidth=0.6, alpha=0.35)

            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Reflectance")
            ax.invert_xaxis()
            ax.grid(True, alpha=0.25)
            ax.legend(loc="best", fontsize=_FONT_SIZE-1)
            plt.tight_layout()

            _current_fig_pk, _current_folder_pk = fig, folder_name
            plt.show()
        except Exception as e:
            print(f"Error: {e}")


def on_run_all_pk(b):
    with status_output_pk:
        clear_output(wait=True)
        folder_name = folder_widget_pk.value
        direction   = peak_direction_pk.value
        if not folder_name or not (BASE_DIR / folder_name).is_dir():
            print("No valid source folder selected.")
            return

        data_dir = BASE_DIR / folder_name
        has_regions = METADATA["has_regions"]
        if has_regions:
            include_parts = {p for p, cb in
                             [("body", body_widget_pk), ("label", label_widget_pk), ("cap", cap_widget_pk)]
                             if cb.value}
            if not include_parts:
                print("Select at least one part (body / label / cap).")
                return
            files = sorted(f for f in data_dir.iterdir()
                           if f.is_file() and any(f.stem.lower().endswith(p) for p in include_parts))
        else:
            files = sorted(f for f in data_dir.iterdir()
                           if f.is_file() and not f.name.startswith("."))

        if not files:
            print("No files match the current folder / part selection.")
            return

        rows, max_peaks, failures = {}, 0, []
        for f in files:
            try:
                wn, y = _read_xy_pk(f)
                peaks, _proms = _detect_peaks_pk(wn, y, direction=direction)
                peak_wns = [round(float(wn[i]), 4) for i in peaks]
                rows[f.stem] = peak_wns
                max_peaks = max(max_peaks, len(peak_wns))
            except Exception as e:
                failures.append(f"{f.name}: {e}")

        if not rows:
            print("No peaks could be extracted from any file.")
            if failures:
                print("Failures:")
                for msg in failures[:10]:
                    print("  -", msg)
            return

        columns = [f"peak_{i+1}_wn" for i in range(max_peaks)]
        table = pd.DataFrame(index=sorted(rows.keys()), columns=columns, dtype=float)
        table.index.name = "filename"
        for fname, peak_wns in rows.items():
            for i, val in enumerate(peak_wns):
                table.iloc[table.index.get_loc(fname), i] = val

        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        direction_tag = "valleys" if direction == "valleys" else "maxima"
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        csv_path = SAVE_DIR / f"{PROJECT_NAME}_Peaks_{direction_tag}_{folder_name}_{timestamp}.csv"
        table.to_csv(csv_path, sep=";")

        _kw_log = _peak_kwargs_pk()
        params_path = SAVE_DIR / f"{PROJECT_NAME}_Peaks_{direction_tag}_{folder_name}_{timestamp}_params.txt"
        _params_lines = [
            "processing_step: peak finding",
            f"timestamp: {datetime.now().isoformat()}",
            f"direction: {direction_tag}",
            f"source_folder: {data_dir}",
            f"output_csv: {csv_path}",
            f"prominence: {prominence_pk.value if (prominence_pk.value and prominence_pk.value > 0) else 'none'}",
            f"height: {height_pk.value if (height_pk.value and height_pk.value > 0) else 'none'}",
            f"distance: {int(distance_pk.value) if (distance_pk.value and distance_pk.value > 1) else 'none'}",
            f"width: {width_pk.value if (width_pk.value and width_pk.value > 0) else 'none'}",
            f"find_peaks_params: {_kw_log if _kw_log else 'none'}",
            "ranking: prominence descending; missing slots = NaN",
            f"parts_included: {sorted(include_parts) if has_regions else 'all'}",
            f"number_of_spectra: {len(table)}",
            f"max_peaks: {max_peaks}",
            f"number_of_failed_files: {len(failures)}",
            f"failed_files: {'; '.join(failures) if failures else 'none'}",
        ]
        params_path.write_text("\n".join(_params_lines) + "\n", encoding="utf-8")

        kw = _peak_kwargs_pk()
        print(f"Peak table exported -> {csv_path}")
        print(f"Params log exported -> {params_path}")
        print(f"  direction: {direction_tag}   spectra: {len(table)}   max peaks: {max_peaks}")
        print(f"  find_peaks params: {kw if kw else '(no constraints)'}")
        print(f"  peaks ranked by prominence (strongest first); missing slots = NaN")
        if failures:
            print(f"  {len(failures)} file(s) failed:")
            for msg in failures[:10]:
                print("   -", msg)
        display(table.head(10))


def on_random_pk(b):
    if file_widget_pk.options:
        import random as _r
        file_widget_pk.value = _r.choice(list(file_widget_pk.options))


random_button_pk.on_click(on_random_pk)
run_button_pk.on_click(on_run_all_pk)
folder_widget_pk.observe(refresh_file_list_pk, names="value")

_pk_watchers = [peak_direction_pk, prominence_pk, height_pk, distance_pk, width_pk]
if METADATA["has_regions"]:
    body_widget_pk.observe(refresh_file_list_pk, names="value")
    label_widget_pk.observe(refresh_file_list_pk, names="value")
    cap_widget_pk.observe(refresh_file_list_pk, names="value")
file_widget_pk.observe(draw_peaks_pk, names="value")
for _w in _pk_watchers:
    _w.observe(draw_peaks_pk, names="value")

_rows_pk = [widgets.HBox([folder_widget_pk]), widgets.HBox([peak_direction_pk])]
if METADATA["has_regions"]:
    _rows_pk.append(widgets.HBox([body_widget_pk, label_widget_pk, cap_widget_pk]))
_rows_pk += [widgets.HBox([file_widget_pk, random_button_pk]),
             widgets.HBox([prominence_pk, height_pk]),
             widgets.HBox([distance_pk, width_pk]),
             widgets.HBox([run_button_pk]), status_output_pk, output_pk]

display(widgets.VBox(_rows_pk))
refresh_file_list_pk()

In [ ]:
# === Cell 12: Class Mean Spectra and Consensus Peaks ===
# Reads a Peak Finder table from SAVE_DIR and its source-folder spectra. Stacks
# region-pooled class mean spectra, marks consensus bands above a prevalence threshold,
# and exports both figures (1/cm and nm) and the per-class band-prevalence table.

from scipy.stats import gaussian_kde

# --- Config ---
_10B_PEAK_GLOB   = f"{PROJECT_NAME}_Peaks_*_*.csv"   # Peak Finder export pattern
_10B_CSV_NAME_RE = re.compile(
    r"^(?P<proj>.+)_Peaks_(?P<direction>valleys|maxima)_(?P<folder>.+)_(?P<ts>\d{8}_\d{6})\.csv$"
)
_10B_CLASSES_ORDER = None   # None -> alphabetical; else explicit list
_10B_PREV_FLOOR    = 0.0     # bands below this are never computed (speed only)

_10b_last = {"fig_wn": None, "fig_wl": None, "bands": None, "meta": None}


# --- Helpers ---
def _10b_list_peak_csvs():
    if not SAVE_DIR.is_dir():
        return []
    files = sorted(SAVE_DIR.glob(_10B_PEAK_GLOB),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    return files


def _10b_source_folder_from_name(csv_name):
    m = _10B_CSV_NAME_RE.match(csv_name)
    return m.group("folder") if m else None


def _10b_read_xy(path):
    df = pd.read_csv(path, sep=";", header=None, names=["wn", "y"])
    df["wn"] = pd.to_numeric(df["wn"], errors="coerce")
    df["y"]  = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna().sort_values("wn")
    return df["wn"].to_numpy(float), df["y"].to_numpy(float)


def _10b_resolve_class(stem):
    cls = get_gt_class(stem)
    if cls is None:
        base, part = parse_filename(stem)
        cls = lookup_gt_class(base, part)
    return cls  # may be None -> dropped


def _10b_load_peaks(csv_path):
    """Return dict[class] -> list of peak-wavenumber arrays (region-pooled)."""
    tbl = pd.read_csv(csv_path, sep=";")
    peak_cols = [c for c in tbl.columns if c.startswith("peak_") and c.endswith("_wn")]
    by_class, dropped = {}, []
    for _, row in tbl.iterrows():
        stem = str(row["filename"]).strip()
        cls = _10b_resolve_class(stem)
        if cls is None:
            dropped.append(stem)
            continue
        vals = [float(row[c]) for c in peak_cols if pd.notna(row[c])]
        by_class.setdefault(cls, []).append(np.asarray(vals, float))
    return by_class, dropped


def _10b_load_class_means(source_folder):
    """Return dict[class] -> (wn_ref, mean_y, std_y) over region-pooled spectra."""
    data_dir = BASE_DIR / source_folder
    if not data_dir.is_dir():
        return {}, f"Source folder not found: {data_dir}"
    per_class = {}
    wn_ref = None
    for f in sorted(data_dir.iterdir()):
        if not f.is_file() or f.name.startswith("."):
            continue
        stem = f.stem
        cls = _10b_resolve_class(stem)
        if cls is None:
            continue
        try:
            wn, y = _10b_read_xy(f)
        except Exception:
            continue
        if wn_ref is None:
            wn_ref = wn
        elif len(wn) != len(wn_ref) or not np.allclose(wn, wn_ref):
            y = np.interp(wn_ref, wn, y)
        per_class.setdefault(cls, []).append(y)
    means = {}
    for cls, ys in per_class.items():
        stack = np.vstack(ys)
        means[cls] = (wn_ref, stack.mean(axis=0), stack.std(axis=0, ddof=0))
    return means, None


def _10b_consensus_bands(peak_arrays, tol):
    """KDE modes over pooled peaks; per-mode prevalence + 5-95 pct spread.
    Returns list of dicts sorted by center. prevalence = frac of spectra with a
    peak within +/- tol of the mode center."""
    pooled = np.concatenate([p for p in peak_arrays if len(p)]) if peak_arrays else np.array([])
    n_spec = len(peak_arrays)
    if pooled.size < 3 or n_spec == 0:
        return []
    lo, hi = 4000.0, 9000.0
    grid = np.arange(lo, hi + 1.0, 2.0)
    bw = max(tol / 1.5, 1.0) / pooled.std(ddof=0) if pooled.std(ddof=0) > 0 else None
    try:
        kde = gaussian_kde(pooled, bw_method=bw)
    except Exception:
        return []
    dens = kde(grid)
    # local maxima
    mode_idx = np.where((dens[1:-1] > dens[:-2]) & (dens[1:-1] >= dens[2:]))[0] + 1
    centers = grid[mode_idx]
    # merge centers closer than tol (keep denser)
    centers = centers[np.argsort(-dens[mode_idx])]
    kept = []
    for c in centers:
        if all(abs(c - k) >= tol for k in kept):
            kept.append(c)
    kept = sorted(kept)
    bands = []
    for c in kept:
        matched_pos, hits = [], 0
        for p in peak_arrays:
            if len(p) == 0:
                continue
            d = np.abs(p - c)
            if d.min() <= tol:
                hits += 1
                matched_pos.append(p[d.argmin()])
        prev = hits / n_spec
        if prev < _10B_PREV_FLOOR or not matched_pos:
            continue
        bands.append({
            "center":     float(np.median(matched_pos)),
            "lo":         float(np.percentile(matched_pos, 5)),
            "hi":         float(np.percentile(matched_pos, 95)),
            "prevalence": float(prev),
            "n_present":  int(hits),
            "n_total":    int(n_spec),
        })
    return sorted(bands, key=lambda b: b["center"])


def _10b_wn_to_nm(wn):
    wn = np.asarray(wn, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(wn != 0, 1e7 / wn, np.nan)


def _10b_peak_label(center_wn, domain):
    """Return the peak center text in the appropriate unit for the domain."""
    if domain == "wavenumber":
        return f"{center_wn:.0f}"
    else:
        return f"{1e7 / center_wn:.0f}"


# --- Widgets ---
_10b_style = {"description_width": "initial"}

_10b_csv = widgets.Dropdown(options=[], description="Peak CSV:",
                            style=_10b_style, layout=widgets.Layout(width="620px"))
_10b_tol = widgets.IntSlider(value=60, min=5, max=200, step=5, description="Tolerance (\u00b1cm\u207b\u00b9):",
                             style=_10b_style, continuous_update=False, layout=widgets.Layout(width="420px"))
_10b_prev = widgets.IntSlider(value=100, min=0, max=100, step=1, description="Peak prevalence \u2265 (%):",
                              style=_10b_style, continuous_update=False, layout=widgets.Layout(width="420px"))
_10b_offset = widgets.FloatText(value=0.0, step=0.5, description="Stack offset (0=auto):",
                                style=_10b_style, layout=widgets.Layout(width="260px"))
_10b_show_std  = widgets.Checkbox(value=True, description="Show std band")
_10b_show_text = widgets.Checkbox(value=True, description="Show peak labels")
_10b_export = widgets.Button(description="Export figures + CSV", button_style="success",
                             icon="download", layout=widgets.Layout(width="220px"))

_10b_status = widgets.Output()
_10b_out    = widgets.Output()


# --- Draw ---
def _10b_ordered_classes(keys):
    if _10B_CLASSES_ORDER:
        return [c for c in _10B_CLASSES_ORDER if c in keys]
    return sorted(keys)


def _10b_compute():
    csv_name = _10b_csv.value
    if not csv_name:
        return None, "No peak CSV selected."
    csv_path = SAVE_DIR / csv_name
    src = _10b_source_folder_from_name(csv_name)
    if src is None:
        return None, f"Cannot parse source folder from name: {csv_name}"
    peaks_by_class, dropped = _10b_load_peaks(csv_path)
    means, err = _10b_load_class_means(src)
    if err:
        return None, err
    classes = _10b_ordered_classes(set(peaks_by_class) & set(means))
    if not classes:
        return None, "No class has both peaks and spectra (check GT / folder)."
    tol = float(_10b_tol.value)
    bands = {c: _10b_consensus_bands(peaks_by_class[c], tol) for c in classes}
    return {"classes": classes, "means": means, "bands": bands,
            "src": src, "csv": csv_name, "tol": tol, "dropped": dropped}, None


def _10b_auto_offset(means, classes):
    spans = []
    for c in classes:
        _, m, _ = means[c]
        spans.append(np.percentile(m, 98) - np.percentile(m, 2))
    return round(1.25 * float(np.median(spans)), 3) if spans else 1.0


def _10b_make_figure(data, domain):
    """domain: 'wavenumber' or 'wavelength'. Returns fig."""
    classes = data["classes"]
    means   = data["means"]
    bands   = data["bands"]
    thr     = _10b_prev.value / 100.0
    offset  = _10b_offset.value if _10b_offset.value > 0 else _10b_auto_offset(means, classes)

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.75))
    for i, c in enumerate(classes):
        wn_ref, m, s = means[c]
        base = i * offset
        color = CLASS_PALETTE.get(c, "#7f7f7f")
        if domain == "wavenumber":
            x = wn_ref
        else:
            x = _10b_wn_to_nm(wn_ref)
        order = np.argsort(x)
        xo, mo, so = x[order], m[order], s[order]
        if _10b_show_std.value:
            ax.fill_between(xo, base + mo - so, base + mo + so, color=color, alpha=0.15, linewidth=0)
        ax.plot(xo, base + mo, color=color, linewidth=1.4)
        # class label pinned to the left axis edge (x in axes frac, y in data)
        ax.text(0.006, base, c, color=color, fontsize=_FONT_SIZE - 1,
                va="center", ha="left", fontweight="bold",
                transform=ax.get_yaxis_transform())
        # consensus peak markers (bands meeting threshold)
        for b in bands[c]:
            if b["prevalence"] < thr:
                continue
            cy = np.interp(b["center"], wn_ref, m)
            mx = b["center"] if domain == "wavenumber" else 1e7 / b["center"]
            ax.scatter([mx], [base + cy], marker="v", s=28, color=color,
                       edgecolor="black", linewidth=0.4, zorder=5)
            if _10b_show_text.value:
                ax.text(mx, base + cy - 0.10 * offset, _10b_peak_label(b["center"], domain),
                        fontsize=_FONT_SIZE - 4, ha="center", va="top",
                        color=color)

    if domain == "wavenumber":
        ax.set_xlabel("Wavenumber (1/cm)")
        ax.invert_xaxis()
    else:
        ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Reflectance")
    ax.set_yticks([])
    ax.grid(True, axis="x", alpha=0.25)
    fig.tight_layout()
    return fig


def _10b_bands_dataframe(data):
    """Per-class band table at the current prevalence threshold (as shown in the figure)."""
    thr = _10b_prev.value / 100.0
    rows = []
    for c in data["classes"]:
        for b in data["bands"][c]:
            if b["prevalence"] < thr:
                continue
            rows.append({
                "class":         c,
                "center_wn":     round(b["center"], 4),
                "lo_wn":         round(b["lo"], 4),
                "hi_wn":         round(b["hi"], 4),
                "center_nm":     round(1e7 / b["center"], 2),
                "lo_nm":         round(1e7 / b["hi"], 2),
                "hi_nm":         round(1e7 / b["lo"], 2),
                "prevalence_pct": round(100 * b["prevalence"], 2),
                "n_present":     b["n_present"],
                "n_total":       b["n_total"],
                "tolerance_wn":  data["tol"],
                "prevalence_threshold_pct": _10b_prev.value,
                "source_folder": data["src"],
                "peak_csv":      data["csv"],
            })
    df = pd.DataFrame(rows, columns=[
        "class", "center_wn", "lo_wn", "hi_wn", "center_nm", "lo_nm", "hi_nm",
        "prevalence_pct", "n_present", "n_total", "tolerance_wn",
        "prevalence_threshold_pct", "source_folder", "peak_csv"])
    return df.sort_values(["class", "center_wn"]).reset_index(drop=True)


def _10b_redraw(*_a):
    with _10b_out:
        clear_output(wait=True)
        plt.close("all")
        data, err = _10b_compute()
        if err:
            print(err); return
        fig_wn = _10b_make_figure(data, "wavenumber")
        plt.show()
        fig_wl = _10b_make_figure(data, "wavelength")
        plt.show()
        _10b_last.update({"fig_wn": fig_wn, "fig_wl": fig_wl,
                          "bands": _10b_bands_dataframe(data), "meta": data})
        n = sum((b["prevalence"] >= _10b_prev.value / 100.0)
                for c in data["classes"] for b in data["bands"][c])
        print(f"Classes: {', '.join(data['classes'])}   |   source: {data['src']}")
        if data["dropped"]:
            print(f"  {len(data['dropped'])} spectrum/spectra had no GT class (excluded):")
            for s in data["dropped"]:
                print(f"    - {s}")
        print(f"  bands shown at \u2265{_10b_prev.value}% prevalence (tol \u00b1{data['tol']:.0f} cm\u207b\u00b9): {n}")


def _10b_on_export(_b):
    with _10b_status:
        clear_output(wait=True)
        if _10b_last["fig_wn"] is None:
            print("Nothing to export \u2014 adjust a control to render first.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        p_wn  = SAVE_DIR / f"{PROJECT_NAME}_MeanSpectra_Wavenumber_{ts}.png"
        p_wl  = SAVE_DIR / f"{PROJECT_NAME}_MeanSpectra_Wavelength_{ts}.png"
        p_csv = SAVE_DIR / f"{PROJECT_NAME}_ClassBandPrevalence_{ts}.csv"
        _10b_last["fig_wn"].savefig(p_wn, dpi=150, bbox_inches="tight")
        _10b_last["fig_wl"].savefig(p_wl, dpi=150, bbox_inches="tight")
        _10b_last["bands"].to_csv(p_csv, sep=";", index=False)
        print(f"Saved:\n  {p_wn}\n  {p_wl}\n  {p_csv}")
        print(f"  CSV rows (bands at \u2265{_10b_prev.value}% prevalence): {len(_10b_last['bands'])}")


def _10b_refresh_csv_list(*_a):
    files = _10b_list_peak_csvs()
    opts = [f.name for f in files]
    _10b_csv.options = opts
    if opts and _10b_csv.value not in opts:
        _10b_csv.value = opts[0]


# --- Wiring ---
_10b_csv.observe(_10b_redraw, names="value")
for _w in (_10b_tol, _10b_prev, _10b_offset, _10b_show_std, _10b_show_text):
    _w.observe(_10b_redraw, names="value")
_10b_export.on_click(_10b_on_export)

_10b_refresh_csv_list()
display(widgets.VBox([
    widgets.HBox([_10b_csv]),
    widgets.HBox([_10b_tol, _10b_prev]),
    widgets.HBox([_10b_offset, _10b_show_std, _10b_show_text]),
    widgets.HBox([_10b_export]),
    _10b_status, _10b_out,
]))
_10b_redraw()


In [ ]:
# === Cell 13: Class Outlier Browser ===
# Flags per-class outliers in a Peak Finder table by core-band match fraction and/or
# peak-count robust z-score. Plots class mean ± std vs. the outlier with missing core
# bands and extra peaks marked (1/cm and nm); exports flagged rows and figures.

from scipy.stats import gaussian_kde

_10C_PEAK_GLOB   = f"{PROJECT_NAME}_Peaks_*_*.csv"
_10C_CSV_NAME_RE = re.compile(
    r"^(?P<proj>.+)_Peaks_(?P<direction>valleys|maxima)_(?P<folder>.+)_(?P<ts>\d{8}_\d{6})\.csv$"
)
_10c_last = {"fig_wn": None, "fig_wl": None, "flagged": None, "meta": None}


# --- Shared helpers ---
def _10c_source_folder_from_name(csv_name):
    m = _10C_CSV_NAME_RE.match(csv_name)
    return m.group("folder") if m else None


def _10c_read_xy(path):
    df = pd.read_csv(path, sep=";", header=None, names=["wn", "y"])
    df["wn"] = pd.to_numeric(df["wn"], errors="coerce")
    df["y"]  = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna().sort_values("wn")
    return df["wn"].to_numpy(float), df["y"].to_numpy(float)


def _10c_resolve_class(stem):
    cls = get_gt_class(stem)
    if cls is None:
        base, part = parse_filename(stem)
        cls = lookup_gt_class(base, part)
    return cls


def _10c_wn_to_nm(wn):
    wn = np.asarray(wn, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(wn != 0, 1e7 / wn, np.nan)


def _10c_load_peaks_rows(csv_path):
    """Return list of dicts: {stem, class, region, peaks(np.array)}."""
    tbl = pd.read_csv(csv_path, sep=";")
    pcols = [c for c in tbl.columns if c.startswith("peak_") and c.endswith("_wn")]
    rows = []
    for _, r in tbl.iterrows():
        stem = str(r["filename"]).strip()
        cls = _10c_resolve_class(stem)
        if cls is None:
            continue
        _, region = parse_filename(stem)
        vals = np.asarray([float(r[c]) for c in pcols if pd.notna(r[c])], float)
        rows.append({"stem": stem, "class": cls, "region": region, "peaks": vals})
    return rows


def _10c_consensus_bands(peak_arrays, tol):
    pooled = np.concatenate([p for p in peak_arrays if len(p)]) if peak_arrays else np.array([])
    n_spec = len(peak_arrays)
    if pooled.size < 3 or n_spec == 0:
        return []
    grid = np.arange(4000.0, 9001.0, 2.0)
    bw = max(tol / 1.5, 1.0) / pooled.std(ddof=0) if pooled.std(ddof=0) > 0 else None
    try:
        kde = gaussian_kde(pooled, bw_method=bw)
    except Exception:
        return []
    dens = kde(grid)
    mode_idx = np.where((dens[1:-1] > dens[:-2]) & (dens[1:-1] >= dens[2:]))[0] + 1
    centers = grid[mode_idx][np.argsort(-dens[mode_idx])]
    kept = []
    for c in centers:
        if all(abs(c - k) >= tol for k in kept):
            kept.append(c)
    kept = sorted(kept)
    bands = []
    for c in kept:
        matched, hits = [], 0
        for p in peak_arrays:
            if len(p) == 0:
                continue
            d = np.abs(p - c)
            if d.min() <= tol:
                hits += 1
                matched.append(p[d.argmin()])
        if matched:
            bands.append({"center": float(np.median(matched)),
                          "prevalence": hits / n_spec})
    return sorted(bands, key=lambda b: b["center"])


def _10c_load_spectra(source_folder):
    """dict[stem] -> (wn, y); plus dict[class] -> list of stems."""
    data_dir = BASE_DIR / source_folder
    spectra, by_class = {}, {}
    if not data_dir.is_dir():
        return spectra, by_class, f"Source folder not found: {data_dir}"
    for f in sorted(data_dir.iterdir()):
        if not f.is_file() or f.name.startswith("."):
            continue
        stem = f.stem
        cls = _10c_resolve_class(stem)
        if cls is None:
            continue
        try:
            wn, y = _10c_read_xy(f)
        except Exception:
            continue
        spectra[stem] = (wn, y)
        by_class.setdefault(cls, []).append(stem)
    return spectra, by_class, None


def _10c_deviation_details(row, core_centers, tol):
    """Return (missing_wns, extra_wns) for one spectrum against its class core."""
    p = row["peaks"]
    missing = [ctr for ctr in core_centers if not (len(p) and np.min(np.abs(p - ctr)) <= tol)]
    if len(core_centers):
        extra = [float(pk) for pk in p if np.min(np.abs(np.asarray(core_centers) - pk)) > tol]
    else:
        extra = [float(pk) for pk in p]
    return missing, extra


# --- Widgets ---
_10c_style = {"description_width": "initial"}
_10c_csv = widgets.Dropdown(options=[], description="Peak CSV:",
                            style=_10c_style, layout=widgets.Layout(width="620px"))
_10c_tol = widgets.IntSlider(value=60, min=5, max=200, step=5, description="Tolerance (\u00b1cm\u207b\u00b9):",
                             style=_10c_style, continuous_update=False, layout=widgets.Layout(width="420px"))
_10c_core = widgets.IntSlider(value=90, min=50, max=100, step=1, description="Core band prevalence \u2265 (%):",
                              style=_10c_style, continuous_update=False, layout=widgets.Layout(width="420px"))
_10c_mfrac = widgets.IntSlider(value=75, min=0, max=100, step=1, description="Match-fraction flag < (%):",
                               style=_10c_style, continuous_update=False, layout=widgets.Layout(width="420px"))
_10c_zthr = widgets.FloatText(value=3.0, step=0.5, description="Peak-count |z| flag >:",
                              style=_10c_style, layout=widgets.Layout(width="260px"))
_10c_pick = widgets.Dropdown(options=[], description="Outlier:",
                             style=_10c_style, layout=widgets.Layout(width="620px"))
_10c_show_missing = widgets.Checkbox(value=True, description="Show missing core bands")
_10c_show_extra   = widgets.Checkbox(value=True, description="Show unmatched extra peaks")
_10c_export = widgets.Button(description="Export figures + CSV", button_style="success",
                             icon="download", layout=widgets.Layout(width="220px"))
_10c_status = widgets.Output()
_10c_out    = widgets.Output()


# --- Detection ---
def _10c_detect():
    csv_name = _10c_csv.value
    if not csv_name:
        return None, "No peak CSV selected."
    csv_path = SAVE_DIR / csv_name
    src = _10c_source_folder_from_name(csv_name)
    if src is None:
        return None, f"Cannot parse source folder from name: {csv_name}"
    rows = _10c_load_peaks_rows(csv_path)
    spectra, by_class, err = _10c_load_spectra(src)
    if err:
        return None, err
    tol = float(_10c_tol.value)
    core_thr = _10c_core.value / 100.0
    mfrac_thr = _10c_mfrac.value / 100.0
    zthr = float(_10c_zthr.value)

    # peaks grouped by class
    peaks_by_class = {}
    for r in rows:
        peaks_by_class.setdefault(r["class"], []).append(r["peaks"])
    # core bands per class
    core = {}
    for c, arrs in peaks_by_class.items():
        core[c] = [b["center"] for b in _10c_consensus_bands(arrs, tol)
                   if b["prevalence"] >= core_thr]
    # per-class peak-count stats (robust z)
    ncount = {}
    for c, arrs in peaks_by_class.items():
        ns = np.array([len(a) for a in arrs], float)
        med = np.median(ns)
        mad = np.median(np.abs(ns - med)) or 1.0
        ncount[c] = (med, mad)

    flagged = []
    for r in rows:
        c = r["class"]
        centers = core.get(c, [])
        p = r["peaks"]
        if centers:
            hit = sum(1 for ctr in centers if len(p) and np.min(np.abs(p - ctr)) <= tol)
            mfrac = hit / len(centers)
        else:
            mfrac = np.nan
        med, mad = ncount[c]
        z = 0.6745 * (len(p) - med) / mad
        by_mfrac = (not np.isnan(mfrac)) and (mfrac < mfrac_thr)
        by_z = abs(z) > zthr
        if by_mfrac or by_z:
            reasons = []
            if by_mfrac:
                reasons.append(f"match {mfrac:.0%}<{mfrac_thr:.0%}")
            if by_z:
                reasons.append(f"|z|={abs(z):.1f}>{zthr:g}")
            missing_wns, extra_wns = _10c_deviation_details(r, centers, tol)
            flagged.append({
                "filename": r["stem"], "class": c, "region": r["region"] or "",
                "n_peaks": len(p), "n_core_bands": len(centers),
                "match_frac_pct": round(100 * mfrac, 1) if not np.isnan(mfrac) else np.nan,
                "peak_count_z": round(float(z), 2),
                "class_median_n": int(med),
                "flagged_by": " & ".join(reasons),
                "missing_core_wns": ";".join(f"{w:.1f}" for w in missing_wns),
                "extra_peak_wns":   ";".join(f"{w:.1f}" for w in extra_wns),
            })
    meta = {"src": src, "csv": csv_name, "tol": tol, "core_thr": core_thr,
            "mfrac_thr": mfrac_thr, "zthr": zthr, "core": core,
            "spectra": spectra, "by_class": by_class,
            "rows_by_stem": {r["stem"]: r for r in rows}}
    return {"flagged": flagged, "meta": meta}, None


def _10c_class_mean_excl(meta, cls, exclude_stem):
    stems = [s for s in meta["by_class"].get(cls, []) if s != exclude_stem]
    if not stems:
        return None
    wn_ref = meta["spectra"][stems[0]][0]
    ys = []
    for s in stems:
        wn, y = meta["spectra"][s]
        if len(wn) != len(wn_ref) or not np.allclose(wn, wn_ref):
            y = np.interp(wn_ref, wn, y)
        ys.append(y)
    stack = np.vstack(ys)
    return wn_ref, stack.mean(axis=0), stack.std(axis=0, ddof=0)


def _10c_make_figure(meta, stem, domain):
    row = meta["rows_by_stem"][stem]
    cls = row["class"]
    color = CLASS_PALETTE.get(cls, "#7f7f7f")
    ref = _10c_class_mean_excl(meta, cls, stem)
    if ref is None or stem not in meta["spectra"]:
        return None
    wn_ref, m, s = ref
    o_wn, o_y = meta["spectra"][stem]
    if len(o_wn) != len(wn_ref) or not np.allclose(o_wn, wn_ref):
        o_y = np.interp(wn_ref, o_wn, o_y)
        o_wn = wn_ref
    tol = meta["tol"]
    centers = meta["core"].get(cls, [])
    p = row["peaks"]

    missing = [ctr for ctr in centers if not (len(p) and np.min(np.abs(p - ctr)) <= tol)]
    if len(centers):
        extra = [float(pk) for pk in p if np.min(np.abs(np.asarray(centers) - pk)) > tol]
    else:
        extra = [float(pk) for pk in p]

    def X(v):
        return np.asarray(v, float) if domain == "wavenumber" else _10c_wn_to_nm(np.asarray(v, float))

    order = np.argsort(X(wn_ref))
    xr = X(wn_ref)[order]

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.55))
    ax.fill_between(xr, (m - s)[order], (m + s)[order], color=color, alpha=0.15, linewidth=0)
    ax.plot(xr, m[order], color=color, linewidth=1.4, label=f"{cls} mean \u00b1 std")
    ax.plot(xr, o_y[order], color="black", linewidth=1.2, alpha=0.9, label=f"{stem}")

    # (a) missing core bands -> open circle on the mean line
    if missing and _10c_show_missing.value:
        mx = X(missing)
        my = np.interp(np.asarray(missing, float), wn_ref, m)
        ax.scatter(mx, my, marker="o", s=55, facecolors="none",
                   edgecolors="#d62728", linewidths=1.4, zorder=6, label="Missing core band")
    # (b) unmatched extra peaks -> X at the outlier's own valley
    if extra and _10c_show_extra.value:
        ex = X(extra)
        ey = np.interp(np.asarray(extra, float), wn_ref, o_y)
        ax.scatter(ex, ey, marker="x", s=55, color="#9467bd",
                   linewidths=1.4, zorder=6, label="Unmatched extra peak")

    if domain == "wavenumber":
        ax.set_xlabel("Wavenumber (1/cm)")
        ax.invert_xaxis()
    else:
        ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Reflectance")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=_FONT_SIZE - 1)
    fig.tight_layout()
    return fig


def _10c_refresh_pick(flagged):
    opts = [(f"{f['filename']}  ({f['class']}) \u2014 {f['flagged_by']}", f["filename"])
            for f in flagged]
    _10c_pick.unobserve(_10c_draw_selected, names="value")
    _10c_pick.options = opts
    _10c_pick.value = opts[0][1] if opts else None
    _10c_pick.observe(_10c_draw_selected, names="value")


def _10c_draw_selected(*_a):
    with _10c_out:
        clear_output(wait=True)
        plt.close("all")
        meta = _10c_last["meta"]
        stem = _10c_pick.value
        if meta is None or stem is None:
            print("No outlier selected."); return
        fig_wn = _10c_make_figure(meta, stem, "wavenumber")
        if fig_wn is None:
            print(f"Cannot plot {stem} (missing spectrum or empty class)."); return
        plt.show()
        fig_wl = _10c_make_figure(meta, stem, "wavelength")
        plt.show()
        _10c_last["fig_wn"], _10c_last["fig_wl"] = fig_wn, fig_wl


def _10c_recompute(*_a):
    with _10c_status:
        clear_output(wait=True)
        res, err = _10c_detect()
        if err:
            print(err)
            _10c_last.update({"flagged": None, "meta": None})
            _10c_pick.options = []
            return
        _10c_last.update({"flagged": res["flagged"], "meta": res["meta"]})
        n = len(res["flagged"])
        print(f"Source: {res['meta']['src']}   |   flagged outliers: {n}")
        print(f"  rules: match-fraction < {_10c_mfrac.value}% of core bands "
              f"(core \u2265{_10c_core.value}%), OR peak-count |z| > {_10c_zthr.value:g}   "
              f"(tol \u00b1{res['meta']['tol']:.0f} cm\u207b\u00b9)")
        for f in res["flagged"]:
            print(f"   \u2022 {f['filename']:26s} {f['class']:5s} "
                  f"n={f['n_peaks']:2d} match={f['match_frac_pct']}%  z={f['peak_count_z']}  "
                  f"[{f['flagged_by']}]")
    _10c_refresh_pick(_10c_last["flagged"] or [])
    _10c_draw_selected()


def _10c_on_export(_b):
    with _10c_status:
        if not _10c_last["flagged"]:
            print("\nNo flagged outliers to export."); return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        m = _10c_last["meta"]
        df = pd.DataFrame(_10c_last["flagged"])
        # attach the identification parameters to every row (self-describing)
        df["tolerance_wn"]             = m["tol"]
        df["core_prevalence_pct"]      = round(100 * m["core_thr"], 1)
        df["match_frac_threshold_pct"] = round(100 * m["mfrac_thr"], 1)
        df["z_threshold"]              = m["zthr"]
        df["source_folder"]            = m["src"]
        df["peak_csv"]                 = m["csv"]
        p_csv = SAVE_DIR / f"{PROJECT_NAME}_Outliers_{ts}.csv"
        df.to_csv(p_csv, sep=";", index=False)
        lines = [f"\nSaved: {p_csv}  ({len(df)} outliers)"]
        # export figures for the currently selected outlier
        if _10c_last["fig_wn"] is not None:
            stem = _10c_pick.value or "unknown"
            p_wn = SAVE_DIR / f"{PROJECT_NAME}_Outlier_{stem}_Wavenumber_{ts}.png"
            p_wl = SAVE_DIR / f"{PROJECT_NAME}_Outlier_{stem}_Wavelength_{ts}.png"
            _10c_last["fig_wn"].savefig(p_wn, dpi=150, bbox_inches="tight")
            _10c_last["fig_wl"].savefig(p_wl, dpi=150, bbox_inches="tight")
            lines.append(f"  {p_wn}")
            lines.append(f"  {p_wl}")
        print("\n".join(lines))


def _10c_refresh_csv_list(*_a):
    files = sorted(SAVE_DIR.glob(_10C_PEAK_GLOB),
                   key=lambda p: p.stat().st_mtime, reverse=True) if SAVE_DIR.is_dir() else []
    opts = [f.name for f in files]
    _10c_csv.options = opts
    if opts and _10c_csv.value not in opts:
        _10c_csv.value = opts[0]


# --- Wiring ---
_10c_csv.observe(_10c_recompute, names="value")
for _w in (_10c_tol, _10c_core, _10c_mfrac, _10c_zthr):
    _w.observe(_10c_recompute, names="value")
_10c_pick.observe(_10c_draw_selected, names="value")
_10c_show_missing.observe(_10c_draw_selected, names="value")
_10c_show_extra.observe(_10c_draw_selected, names="value")
_10c_export.on_click(_10c_on_export)

_10c_refresh_csv_list()
display(widgets.VBox([
    widgets.HBox([_10c_csv]),
    widgets.HBox([_10c_tol, _10c_core]),
    widgets.HBox([_10c_mfrac, _10c_zthr]),
    widgets.HBox([_10c_pick]),
    widgets.HBox([_10c_show_missing, _10c_show_extra, _10c_export]),
    _10c_status, _10c_out,
]))
_10c_recompute()


In [ ]:
# === Cell 14: PCA Loadings Viewer ===
# One loading line per PC over the source folder's reference wavenumber axis.

_subfolders_ld = get_subfolders(BASE_DIR)
_default_ld = DEFAULT_FOLDER if DEFAULT_FOLDER in _subfolders_ld else (_subfolders_ld[0] if _subfolders_ld else "")

folder_widget_ld = widgets.Dropdown(
    options=_subfolders_ld, value=_default_ld, description="Source folder:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
)
body_widget_ld  = widgets.Checkbox(value=True, description="Body")
label_widget_ld = widgets.Checkbox(value=True, description="Label")
cap_widget_ld   = widgets.Checkbox(value=True, description="Cap")

n_pcs_widget_ld = widgets.BoundedIntText(
    value=5, min=1, max=20, description="Number of PCs:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="220px"),
)
save_button_ld = widgets.Button(description="Save PNG", button_style="info", tooltip="Save current plot as PNG")

output_ld        = widgets.Output()
status_output_ld = widgets.Output()

_current_fig_ld    = None
_current_folder_ld = None

_PC_CMAP_LD = plt.get_cmap("tab10")


def draw_loadings_ld(*_args):
    global _current_fig_ld, _current_folder_ld

    folder_name = folder_widget_ld.value
    n_pcs       = n_pcs_widget_ld.value
    has_regions = METADATA["has_regions"]

    include_parts = None
    if has_regions:
        include_parts = [p for p, cb in
                         [("body", body_widget_ld), ("label", label_widget_ld), ("cap", cap_widget_ld)]
                         if cb.value]

    with output_ld:
        clear_output(wait=True)
        if not folder_name:
            print(f"No subfolders found in {BASE_DIR}")
            return
        try:
            plt.close("all")
            X, meta, wn_ref = load_spectra_matrix(BASE_DIR / folder_name, include_parts)

            n_samples, n_features = X.shape
            max_pcs = min(n_pcs, n_samples, n_features)
            if max_pcs < n_pcs:
                print(f"Note: requested {n_pcs} PCs, but only {max_pcs} are available "
                      f"(n_samples={n_samples}, n_features={n_features}).")

            pca = PCA(n_components=max_pcs)
            pca.fit(X)
            loadings = pca.components_          # shape (max_pcs, n_features)
            evr = pca.explained_variance_ratio_

            fig, ax = plt.subplots(figsize=(_FW, _FW * 0.5))
            for i in range(max_pcs):
                ax.plot(wn_ref, loadings[i], color=_PC_CMAP_LD(i % 10), linewidth=1.4,
                        label=f"PC{i+1} ({evr[i]*100:.1f}%)")

            ax.axhline(0, color="black", linewidth=0.6, alpha=0.5)
            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Loading Value")
            ax.invert_xaxis()
            ax.grid(True, alpha=0.25)
            ax.legend(loc="best", fontsize=_FONT_SIZE-1)
            plt.tight_layout()

            _current_fig_ld, _current_folder_ld = fig, folder_name
            plt.show()
        except Exception as e:
            print(f"Error: {e}")


def on_save_ld(b):
    with status_output_ld:
        clear_output(wait=True)
        if _current_fig_ld is None:
            print("No plot to save.")
            return
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        png_path = SAVE_DIR / f"{PROJECT_NAME}_PCALoadings_{_current_folder_ld}_{n_pcs_widget_ld.value}PCs.png"
        _current_fig_ld.savefig(png_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {png_path}")

save_button_ld.on_click(on_save_ld)

_watchers_ld = [folder_widget_ld, n_pcs_widget_ld]
if METADATA["has_regions"]:
    _watchers_ld += [body_widget_ld, label_widget_ld, cap_widget_ld]
for w in _watchers_ld:
    w.observe(draw_loadings_ld, names="value")

_rows_ld = [widgets.HBox([folder_widget_ld])]
if METADATA["has_regions"]:
    _rows_ld.append(widgets.HBox([body_widget_ld, label_widget_ld, cap_widget_ld]))
_rows_ld.append(widgets.HBox([n_pcs_widget_ld, save_button_ld]))

display(widgets.VBox(_rows_ld + [status_output_ld, output_ld]))
draw_loadings_ld()

In [ ]:
# === Cell 15: PCA Loadings over Group Spectra ===
# Group Spectra View filters (wavenumber panel only) with selected PCA loadings drawn
# as grouped tab10 bars in front of the spectra. PCA is fit on the full source folder
# (Region filter applied). Does not require the Group Spectra View cell.

import random as _pl_random

display(HTML("""
<style>
.pl-title { font-weight: 700; color: #1f2937; margin: 0 0 6px 0; font-size: 16px; }
.pl-subtitle { color: #4b5563; margin: 0 0 10px 0; font-size: 12px; }
.pl-status { padding: 6px 10px; border-radius: 6px; border: 1px solid #d1d5db;
             background: #f8fafc; color: #111827; font-size: 12px; }
</style>
"""))

_PL_STYLE = {"description_width": "initial"}
_PL_FNAME_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_number>\d+)_(?P<region>[A-Za-z0-9]+)$")
_PL_SIMPLE_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_")

_PL_PC_CMAP = plt.get_cmap("tab10")

pl_last_figure = {"fig": None, "path": None}


def _pl_discover_folders():
    options = []
    if BASE_DIR.exists():
        for sf in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()], key=lambda p: p.name):
            options.append((sf.name, str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options


def _pl_read_spectrum(path):
    from io import StringIO
    path = Path(path)
    raw_text = path.read_text(encoding="utf-8", errors="replace")
    lines = raw_text.strip().splitlines()
    data_lines = [l for l in lines if not l.startswith("##")]
    if not data_lines:
        raise ValueError("File is empty after stripping headers.")
    block = "\n".join(data_lines)
    sep = ";" if ";" in data_lines[0] else r"\s+"
    df = pd.read_csv(StringIO(block), sep=sep, header=None,
                     names=["wavenumber", "signal"], engine="python")
    df["wavenumber"] = pd.to_numeric(df["wavenumber"], errors="coerce")
    df["signal"] = pd.to_numeric(df["signal"], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    if df.empty:
        raise ValueError("File contains no valid numeric rows.")
    return df


def _pl_folder_metadata(folder):
    folder = Path(folder)
    rows = []
    has_regions = METADATA["has_regions"]
    gt_lookup = METADATA["gt_lookup"]
    if not folder.exists():
        return pd.DataFrame(columns=["filename", "material", "object_number", "region", "path", "gt_class"])
    for p in sorted([f for f in folder.iterdir() if f.is_file() and not f.name.startswith(".")],
                    key=lambda p: p.name):
        stem = p.stem
        if has_regions:
            m = _PL_FNAME_RE.match(stem)
            if m is None:
                continue
            info = m.groupdict()
            gt = gt_lookup.get(stem)
            if gt is None:
                base_name = f"{info['material']}_obj_{info['object_number']}"
                gt = lookup_gt_class(base_name, info["region"])
            rows.append({"filename": stem, "material": info["material"],
                         "object_number": info["object_number"], "region": info["region"],
                         "path": str(p), "gt_class": gt if gt else "unknown"})
        else:
            m = _PL_SIMPLE_RE.match(stem)
            material = m.group("material") if m else "unknown"
            rows.append({"filename": stem, "material": material, "object_number": "0",
                         "region": "n/a", "path": str(p),
                         "gt_class": gt_lookup.get(stem, "unknown")})
    return pd.DataFrame(rows)


def _pl_filter(df, materials, regions, objects, filename, gt_classes):
    out = df
    def _apply(o, col, sel):
        if sel:
            return o[o[col].astype(str).isin([str(x) for x in sel])]
        return o
    out = _apply(out, "material", materials)
    out = _apply(out, "region", regions)
    out = _apply(out, "object_number", objects)
    out = _apply(out, "gt_class", gt_classes)
    if filename != "All":
        out = out[out["filename"] == filename]
    return out.reset_index(drop=True)


def _pl_align(dataframes):
    ref_x = dataframes[0]["wavenumber"].to_numpy(dtype=float)
    aligned = []
    for df in dataframes:
        x = df["wavenumber"].to_numpy(dtype=float)
        y = df["signal"].to_numpy(dtype=float)
        if len(x) == len(ref_x) and np.allclose(x, ref_x):
            aligned.append(y)
            continue
        descending = np.any(np.diff(x) < 0)
        x_asc = x[::-1] if descending else x
        y_asc = y[::-1] if descending else y
        ref_desc = np.any(np.diff(ref_x) < 0)
        ref_asc = ref_x[::-1] if ref_desc else ref_x
        interp_y = np.interp(ref_asc, x_asc, y_asc)
        aligned.append(interp_y[::-1] if ref_desc else interp_y)
    return ref_x, np.vstack(aligned)


def _pl_color(label, idx, cmap):
    c = CLASS_PALETTE.get(str(label))
    return c if c is not None else cmap(idx)


def _pl_auto_offset(y_list):
    spans = [np.percentile(y, 98) - np.percentile(y, 2) for y in y_list if len(y)]
    return round(1.25 * float(np.median(spans)), 3) if spans else 1.0


def _pl_stacked_label(ax, base, text, color):
    ax.text(0.006, base, text, color=color, fontsize=_FONT_SIZE - 1,
            va="center", ha="left", fontweight="bold", zorder=6,
            transform=ax.get_yaxis_transform())


def _pl_build_groups(df, grouping_mode):
    groups = {}
    if grouping_mode == "by material":
        for name, gdf in df.groupby("material"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by region":
        for name, gdf in df.groupby("region"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by object":
        for name, gdf in df.groupby("object_number"):
            groups[f"obj_{name}"] = gdf.reset_index(drop=True)
    elif grouping_mode == "by GT class":
        col = "gt_class" if "gt_class" in df.columns else "material"
        for name, gdf in df.groupby(col):
            groups[str(name)] = gdf.reset_index(drop=True)
    else:
        groups["all"] = df.reset_index(drop=True)
    return groups


def _pl_fit_loadings(folder_path, region_filter, n_pcs):
    """Fit PCA on the full source folder (optionally region-filtered) and
    return (wn_ref, loadings[n_pcs, n_features], explained_variance_ratio, max_pcs)."""
    include_parts = None
    if METADATA["has_regions"] and region_filter:
        include_parts = list(region_filter)
    X, meta, wn_ref = load_spectra_matrix(Path(folder_path), include_parts)
    max_pcs = min(n_pcs, X.shape[0], X.shape[1])
    pca = PCA(n_components=max_pcs)
    pca.fit(X)
    return wn_ref, pca.components_, pca.explained_variance_ratio_, max_pcs


def pl_save_plot(_):
    fig = pl_last_figure.get("fig")
    if fig is None:
        pl_status.value = '<div class="pl-status">No plot is currently available to save.</div>'
        return
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = SAVE_DIR / f"{PROJECT_NAME}_LoadingsOverlay_{timestamp}.png"
    try:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        pl_last_figure["path"] = str(out_path)
        pl_status.value = f'<div class="pl-status">Plot saved to <b>{out_path}</b></div>'
    except Exception as exc:
        pl_status.value = f'<div class="pl-status">Failed to save plot: {exc}</div>'


def pl_update(_=None):
    pl_plot_output.clear_output(wait=True)
    pl_last_figure["fig"] = None
    pl_last_figure["path"] = None

    folder_path = pl_source_dropdown.value
    if folder_path is None:
        pl_status.value = '<div class="pl-status">No source folder selected.</div>'
        return

    folder_name = Path(folder_path).name
    meta_df = _pl_folder_metadata(folder_path)
    if meta_df.empty:
        pl_status.value = f'<div class="pl-status">No parseable spectra found in {folder_name}.</div>'
        return

    catalog = _pl_filter(meta_df, materials=pl_material.value, regions=pl_region.value,
                         objects=pl_object.value, filename=pl_filename.value,
                         gt_classes=pl_gt_class.value)
    if catalog.empty:
        pl_status.value = '<div class="pl-status">No spectra match the selected filters.</div>'
        return

    _ensure_palette(catalog["gt_class"].unique())

    try:
        # Fit PCA loadings on the full folder (region-filtered), independent of
        # the material/object/GT filter used for the spectra lines.
        wn_ref, loadings, evr, max_pcs = _pl_fit_loadings(folder_path, pl_region.value, pl_n_pcs.value)

        selected_pcs = sorted(set(int(v) for v in pl_pc_select.value if int(v) <= max_pcs))
        if not selected_pcs:
            selected_pcs = [1]
        dropped = sorted(set(int(v) for v in pl_pc_select.value) - set(selected_pcs))

        with pl_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(_FW, _FW * 0.55), constrained_layout=True)
            ax_bar = ax.twinx()   # drawn after ax, so it renders on top

            # --- De-emphasised spectra (thin lines) ---
            warnings = []
            _stacked = pl_layout.value == "stacked"
            if pl_mode.value == "single":
                cmap = plt.get_cmap("tab20", max(len(catalog), 1))
                _prepared = []
                for idx, (_, row) in enumerate(catalog.iterrows()):
                    try:
                        df = _pl_read_spectrum(row["path"])
                    except Exception as exc:
                        warnings.append(f'{Path(row["path"]).name} failed ({exc})')
                        continue
                    wn = df["wavenumber"].to_numpy(dtype=float)
                    y = df["signal"].to_numpy(dtype=float)
                    color = _pl_color(row.get("gt_class", "unknown"), idx, cmap)
                    label = f'{row.get("gt_class", "unknown")} | {row["filename"]}'
                    _prepared.append((wn, y, color, label))
                if _stacked and _prepared:
                    _off = pl_offset.value if pl_offset.value > 0 else _pl_auto_offset([p[1] for p in _prepared])
                    for _i, (wn, y, color, label) in enumerate(_prepared):
                        _base = _i * _off
                        ax.plot(wn, y + _base, color=color, linewidth=0.7, alpha=0.75, zorder=2)
                        _pl_stacked_label(ax, _base, label, color)
                else:
                    for wn, y, color, label in _prepared:
                        ax.plot(wn, y, color=color, linewidth=0.7, alpha=0.75, label=label, zorder=2)
                n_groups = len(catalog)
            else:
                groups = _pl_build_groups(catalog, pl_grouping.value)
                cmap = plt.get_cmap("tab10", max(len(groups), 1))
                _prepared = []
                for idx, (legend_label, group_df) in enumerate(groups.items()):
                    spectra_list = []
                    for _, row in group_df.iterrows():
                        try:
                            spectra_list.append(_pl_read_spectrum(row["path"]))
                        except Exception as exc:
                            warnings.append(f'{Path(row["path"]).name} failed ({exc})')
                    if not spectra_list:
                        continue
                    ref_x, stack = _pl_align(spectra_list)
                    mean_y, std_y = stack.mean(axis=0), stack.std(axis=0, ddof=0)
                    if pl_grouping.value == "all":
                        line_color, fill_color, fill_alpha = "black", "lightgrey", 0.35
                    else:
                        line_color = fill_color = _pl_color(legend_label, idx, cmap)
                        fill_alpha = 0.10
                    _prepared.append((ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label))
                if _stacked and _prepared:
                    _off = pl_offset.value if pl_offset.value > 0 else _pl_auto_offset([p[1] for p in _prepared])
                    for _i, (ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label) in enumerate(_prepared):
                        _base = _i * _off
                        ax.plot(ref_x, mean_y + _base, color=line_color, linewidth=1.0, alpha=0.9, zorder=2)
                        if pl_show_std.value:
                            ax.fill_between(ref_x, mean_y + _base - std_y, mean_y + _base + std_y, color=fill_color, alpha=fill_alpha, zorder=2)
                        _pl_stacked_label(ax, _base, legend_label, line_color)
                else:
                    for ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label in _prepared:
                        ax.plot(ref_x, mean_y, color=line_color, linewidth=1.0, alpha=0.9, label=legend_label, zorder=2)
                        if pl_show_std.value:
                            ax.fill_between(ref_x, mean_y - std_y, mean_y + std_y, color=fill_color, alpha=fill_alpha, zorder=2)
                n_groups = len(groups)

                
            # --- Loading bars in front, one sub-series per selected PC ---
            wn_sorted = np.sort(wn_ref)
            base_spacing = np.median(np.abs(np.diff(wn_sorted))) if len(wn_sorted) > 1 else 1.0
            n_sel = len(selected_pcs)
            group_width = base_spacing * 0.9
            sub_width = group_width / n_sel

            for j, pc_num in enumerate(selected_pcs):
                pc_idx = pc_num - 1
                offset = (j - (n_sel - 1) / 2) * sub_width
                ax_bar.bar(wn_ref + offset, loadings[pc_idx], width=sub_width,
                          color=_PL_PC_CMAP(pc_idx % 10),
                          alpha=0.85, zorder=5,
                          label=f"PC{pc_num} Loading ({evr[pc_idx]*100:.1f}%)")
            ax_bar.axhline(0, color="#7f7f7f", linewidth=0.6, alpha=0.6, zorder=4)
            ax_bar.set_ylabel("Loading Value")

            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Reflectance")
            ax.grid(True, alpha=0.25)
            ax.invert_xaxis()
            if _stacked:
                ax.set_yticks([])

            # combined legend, inside the axes
            h_spec, l_spec = ax.get_legend_handles_labels()
            h_bar, l_bar = ax_bar.get_legend_handles_labels()
            ax.legend(h_spec + h_bar, l_spec + l_bar, loc="best", fontsize=_FONT_SIZE - 1)

            pl_last_figure["fig"] = fig
            plt.show()

        pc_label = ", ".join(f"PC{p}" for p in selected_pcs)
        dropped_text = "" if not dropped else f" (PC{', PC'.join(str(d) for d in dropped)} unavailable — only {max_pcs} PCs fit)"
        warning_text = "" if not warnings else " Warnings: " + "; ".join(warnings[:8])
        pl_status.value = (f'<div class="pl-status">Updated with <b>{n_groups}</b> series '
                           f'from {folder_name}, overlaying {pc_label}{dropped_text}.{warning_text}</div>')
    except Exception as exc:
        pl_last_figure["fig"] = None
        pl_status.value = f'<div class="pl-status">Visualization update failed: {exc}</div>'


def pl_refresh_filters(_=None):
    folder_path = pl_source_dropdown.value
    has_regions = METADATA["has_regions"]
    if folder_path is None:
        for w in (pl_material, pl_region, pl_gt_class, pl_object):
            w.options = []
            w.value = ()
        pl_filename.options = ["All"]
        pl_filename.value = "All"
        return

    meta_df = _pl_folder_metadata(folder_path)
    if meta_df.empty:
        for w in (pl_material, pl_region, pl_gt_class, pl_object):
            w.options = []
            w.value = ()
        pl_filename.options = ["All"]
        pl_filename.value = "All"
        pl_update()
        return

    materials  = sorted(meta_df["material"].dropna().astype(str).unique().tolist())
    gt_classes = sorted(meta_df["gt_class"].dropna().astype(str).unique().tolist())
    objects    = sorted(meta_df["object_number"].dropna().astype(str).unique().tolist())

    pl_material.options = materials
    pl_material.value = tuple(v for v in pl_material.value if v in materials)

    if has_regions:
        regions = sorted(meta_df["region"].dropna().astype(str).unique().tolist())
        pl_region.options = regions
        pl_region.disabled = False
        pl_region.value = tuple(v for v in pl_region.value if v in regions)
    else:
        pl_region.options = []
        pl_region.value = ()
        pl_region.disabled = True

    pl_object.options = objects
    pl_object.value = tuple(v for v in pl_object.value if v in objects)

    pl_gt_class.options = gt_classes
    pl_gt_class.value = tuple(v for v in pl_gt_class.value if v in gt_classes)

    filtered = _pl_filter(meta_df, materials=pl_material.value, regions=pl_region.value,
                          objects=pl_object.value, filename="All", gt_classes=pl_gt_class.value)
    filenames = ["All"] + sorted(filtered["filename"].dropna().astype(str).unique().tolist())
    pl_filename.options = filenames
    if pl_filename.value not in filenames:
        pl_filename.value = "All"

    if has_regions:
        pl_grouping.options = ["all", "by material", "by region", "by object", "by GT class"]
    else:
        pl_grouping.options = ["all", "by material", "by object", "by GT class"]
        if pl_grouping.value == "by region":
            pl_grouping.value = "by material"

    pl_update()


def pl_random_file(_):
    options = [opt for opt in pl_filename.options if opt != "All"]
    if options:
        pl_filename.value = _pl_random.choice(options)


def pl_refresh_folders(_=None):
    pl_source_dropdown.options = _pl_discover_folders()
    pl_refresh_filters()


def pl_sync_pc_options(_=None):
    """Keep the 'Show PC #' options in sync with the 'PCA fit — # PCs' count."""
    n = pl_n_pcs.value
    options = list(range(1, n + 1))
    prev = tuple(v for v in pl_pc_select.value if v in options)
    pl_pc_select.options = options
    pl_pc_select.value = prev if prev else tuple(options[:min(5, n)])


# --- Widgets ---
_pl_folder_options = _pl_discover_folders()
pl_source_dropdown = widgets.Dropdown(
    options=_pl_folder_options,
    value=_pl_folder_options[0][1] if _pl_folder_options else None,
    description="Source folder", style=_PL_STYLE, layout=widgets.Layout(width="500px"))

_pl_sel_layout = widgets.Layout(width="200px", height="120px")
pl_material = widgets.SelectMultiple(options=[], value=(), description="Material", style=_PL_STYLE, layout=_pl_sel_layout)
pl_region   = widgets.SelectMultiple(options=[], value=(), description="Region", style=_PL_STYLE, layout=_pl_sel_layout,
                                     disabled=not METADATA["has_regions"])
pl_gt_class = widgets.SelectMultiple(options=[], value=(), description="GT class", style=_PL_STYLE, layout=_pl_sel_layout)
pl_object   = widgets.SelectMultiple(options=[], value=(), description="Object number", style=_PL_STYLE, layout=_pl_sel_layout)
pl_filename = widgets.Dropdown(options=["All"], value="All", description="Specific filename", style=_PL_STYLE, layout=widgets.Layout(width="340px"))
pl_mode = widgets.ToggleButtons(options=[("Single View", "single"), ("Aggregated Group View", "aggregated")], value="single", description="Mode", style=_PL_STYLE)

_pl_grouping_opts = ["all", "by material", "by region", "by object", "by GT class"] if METADATA["has_regions"] else ["all", "by material", "by object", "by GT class"]
pl_grouping = widgets.Dropdown(options=_pl_grouping_opts, value="by material", description="Group by", style=_PL_STYLE, layout=widgets.Layout(width="220px"))
pl_layout = widgets.ToggleButtons(options=[("Overlaid", "overlaid"), ("Stacked", "stacked")],
                                  value="overlaid", description="Layout", style=_PL_STYLE)
pl_offset = widgets.FloatText(value=0.0, step=0.5, description="Stack offset (0=auto)",
                              style=_PL_STYLE, layout=widgets.Layout(width="240px"))
pl_show_std = widgets.Checkbox(value=True, description="Show std band")

pl_n_pcs = widgets.BoundedIntText(value=5, min=1, max=20, description="PCA fit — # PCs:",
                                  style=_PL_STYLE, layout=widgets.Layout(width="220px"))
pl_pc_select = widgets.SelectMultiple(options=list(range(1, 6)), value=(1, 2, 3, 4, 5),
                                      description="Show PC(s):", style=_PL_STYLE,
                                      layout=widgets.Layout(width="160px", height="120px"))

pl_random_button  = widgets.Button(description="Random", icon="random", layout=widgets.Layout(width="150px", height="34px"))
pl_refresh_button = widgets.Button(description="Refresh folders", icon="refresh", button_style="info", layout=widgets.Layout(width="150px", height="34px"))
pl_save_button    = widgets.Button(description="Save Plot", icon="save", button_style="success", layout=widgets.Layout(width="150px", height="34px"))
pl_status = widgets.HTML('<div class="pl-status">Select a source folder to inspect spectra.</div>')
pl_plot_output = widgets.Output(layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="8px", height="620px"))

pl_source_dropdown.observe(pl_refresh_filters, names="value")
pl_refresh_button.on_click(pl_refresh_folders)
pl_random_button.on_click(pl_random_file)
pl_save_button.on_click(pl_save_plot)
pl_n_pcs.observe(pl_sync_pc_options, names="value")
for w in (pl_material, pl_region, pl_object, pl_gt_class):
    w.observe(pl_refresh_filters, names="value")
for w in (pl_filename, pl_mode, pl_grouping, pl_pc_select, pl_layout, pl_offset, pl_show_std):
    w.observe(pl_update, names="value")

pl_controls = widgets.VBox([
    widgets.HBox([pl_source_dropdown, pl_refresh_button]),
    widgets.HBox([pl_material, pl_region, pl_gt_class, pl_object]),
    widgets.HBox([pl_filename, pl_random_button]),
    widgets.HBox([pl_mode, pl_grouping]),
    widgets.HBox([pl_layout, pl_offset, pl_show_std]),
    widgets.HBox([pl_n_pcs, pl_pc_select]),
    widgets.HBox([pl_save_button]),
])

pl_panel = widgets.VBox([
    widgets.HTML(
        '<div class="pl-title">PCA Loadings Overlay on Group Spectra</div>'
        '<div class="pl-subtitle">Same filters as Group Spectra View. The PCA is fit on the full '
        'source folder (respecting the Region filter, independent of Material/Object/GT filters) '
        'with "PCA fit — # PCs" components; "Show PC(s)" multi-selects which loading vectors are '
        'drawn as bold grouped bars in front of the (thin, subdued) spectra lines.</div>'
    ),
    pl_controls, pl_status, pl_plot_output,
], layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="12px", margin="8px 0"))

display(pl_panel)
pl_refresh_filters()

In [ ]:
# === Cell 16: PCA Loading Bands over Spectra ===
# Group Spectra View filters (wavenumber panel only) with a tab10 band per PC wherever
# |loading| >= threshold * max|loading|. PCA is fit on the full source folder (Region
# filter applied). Does not require the Group Spectra View cell.

import random as _pl_random
from matplotlib.patches import Patch

display(HTML("""
<style>
.pl-title { font-weight: 700; color: #1f2937; margin: 0 0 6px 0; font-size: 16px; }
.pl-subtitle { color: #4b5563; margin: 0 0 10px 0; font-size: 12px; }
.pl-status { padding: 6px 10px; border-radius: 6px; border: 1px solid #d1d5db;
             background: #f8fafc; color: #111827; font-size: 12px; }
</style>
"""))

_PL_STYLE = {"description_width": "initial"}
_PL_FNAME_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_number>\d+)_(?P<region>[A-Za-z0-9]+)$")
_PL_SIMPLE_RE = re.compile(r"^(?P<material>[A-Za-z0-9]+)_")
_PL_PC_CMAP = plt.get_cmap("tab10")

pl_last_figure = {"fig": None, "path": None}


def _pl_discover_folders():
    options = []
    if BASE_DIR.exists():
        for sf in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()], key=lambda p: p.name):
            options.append((sf.name, str(sf)))
    if not options:
        options.append(("<No folders found>", None))
    return options


def _pl_read_spectrum(path):
    from io import StringIO
    path = Path(path)
    raw_text = path.read_text(encoding="utf-8", errors="replace")
    lines = raw_text.strip().splitlines()
    data_lines = [l for l in lines if not l.startswith("##")]
    if not data_lines:
        raise ValueError("File is empty after stripping headers.")
    block = "\n".join(data_lines)
    sep = ";" if ";" in data_lines[0] else r"\s+"
    df = pd.read_csv(StringIO(block), sep=sep, header=None,
                     names=["wavenumber", "signal"], engine="python")
    df["wavenumber"] = pd.to_numeric(df["wavenumber"], errors="coerce")
    df["signal"] = pd.to_numeric(df["signal"], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    if df.empty:
        raise ValueError("File contains no valid numeric rows.")
    return df


def _pl_folder_metadata(folder):
    folder = Path(folder)
    rows = []
    has_regions = METADATA["has_regions"]
    gt_lookup = METADATA["gt_lookup"]
    if not folder.exists():
        return pd.DataFrame(columns=["filename", "material", "object_number", "region", "path", "gt_class"])
    for p in sorted([f for f in folder.iterdir() if f.is_file() and not f.name.startswith(".")],
                    key=lambda p: p.name):
        stem = p.stem
        if has_regions:
            m = _PL_FNAME_RE.match(stem)
            if m is None:
                continue
            info = m.groupdict()
            gt = gt_lookup.get(stem)
            if gt is None:
                base_name = f"{info['material']}_obj_{info['object_number']}"
                gt = lookup_gt_class(base_name, info["region"])
            rows.append({"filename": stem, "material": info["material"],
                         "object_number": info["object_number"], "region": info["region"],
                         "path": str(p), "gt_class": gt if gt else "unknown"})
        else:
            m = _PL_SIMPLE_RE.match(stem)
            material = m.group("material") if m else "unknown"
            rows.append({"filename": stem, "material": material, "object_number": "0",
                         "region": "n/a", "path": str(p),
                         "gt_class": gt_lookup.get(stem, "unknown")})
    return pd.DataFrame(rows)


def _pl_filter(df, materials, regions, objects, filename, gt_classes):
    out = df
    def _apply(o, col, sel):
        if sel:
            return o[o[col].astype(str).isin([str(x) for x in sel])]
        return o
    out = _apply(out, "material", materials)
    out = _apply(out, "region", regions)
    out = _apply(out, "object_number", objects)
    out = _apply(out, "gt_class", gt_classes)
    if filename != "All":
        out = out[out["filename"] == filename]
    return out.reset_index(drop=True)


def _pl_align(dataframes):
    ref_x = dataframes[0]["wavenumber"].to_numpy(dtype=float)
    aligned = []
    for df in dataframes:
        x = df["wavenumber"].to_numpy(dtype=float)
        y = df["signal"].to_numpy(dtype=float)
        if len(x) == len(ref_x) and np.allclose(x, ref_x):
            aligned.append(y)
            continue
        descending = np.any(np.diff(x) < 0)
        x_asc = x[::-1] if descending else x
        y_asc = y[::-1] if descending else y
        ref_desc = np.any(np.diff(ref_x) < 0)
        ref_asc = ref_x[::-1] if ref_desc else ref_x
        interp_y = np.interp(ref_asc, x_asc, y_asc)
        aligned.append(interp_y[::-1] if ref_desc else interp_y)
    return ref_x, np.vstack(aligned)


def _pl_color(label, idx, cmap):
    c = CLASS_PALETTE.get(str(label))
    return c if c is not None else cmap(idx)


def _pl_auto_offset(y_list):
    spans = [np.percentile(y, 98) - np.percentile(y, 2) for y in y_list if len(y)]
    return round(1.25 * float(np.median(spans)), 3) if spans else 1.0


def _pl_stacked_label(ax, base, text, color):
    ax.text(0.006, base, text, color=color, fontsize=_FONT_SIZE - 1,
            va="center", ha="left", fontweight="bold", zorder=6,
            transform=ax.get_yaxis_transform())


def _pl_build_groups(df, grouping_mode):
    groups = {}
    if grouping_mode == "by material":
        for name, gdf in df.groupby("material"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by region":
        for name, gdf in df.groupby("region"):
            groups[str(name)] = gdf.reset_index(drop=True)
    elif grouping_mode == "by object":
        for name, gdf in df.groupby("object_number"):
            groups[f"obj_{name}"] = gdf.reset_index(drop=True)
    elif grouping_mode == "by GT class":
        col = "gt_class" if "gt_class" in df.columns else "material"
        for name, gdf in df.groupby(col):
            groups[str(name)] = gdf.reset_index(drop=True)
    else:
        groups["all"] = df.reset_index(drop=True)
    return groups


def _pl_fit_loadings(folder_path, region_filter, n_pcs):
    """Fit PCA on the full source folder (optionally region-filtered) and
    return (wn_ref, loadings[n_pcs, n_features], explained_variance_ratio, max_pcs)."""
    include_parts = None
    if METADATA["has_regions"] and region_filter:
        include_parts = list(region_filter)
    X, meta, wn_ref = load_spectra_matrix(Path(folder_path), include_parts)
    max_pcs = min(n_pcs, X.shape[0], X.shape[1])
    pca = PCA(n_components=max_pcs)
    pca.fit(X)
    return wn_ref, pca.components_, pca.explained_variance_ratio_, max_pcs


def _pl_threshold_spans(wn_ref, loading_vec, threshold_frac):
    """
    Return a list of (x_start, x_end) spans (in wavenumber space) where
    |loading| >= threshold_frac * max|loading|. Boundaries sit at the
    midpoints between consecutive wavenumber samples so adjacent spans
    tile without gaps or overlaps.
    """
    order = np.argsort(wn_ref)
    x_sorted = wn_ref[order]
    v_sorted = np.abs(loading_vec[order])
    peak = v_sorted.max()
    if peak <= 0:
        return []
    mask = v_sorted >= threshold_frac * peak

    # midpoints between consecutive samples -> band edges
    if len(x_sorted) > 1:
        mids = (x_sorted[:-1] + x_sorted[1:]) / 2.0
        left_edges = np.concatenate(([x_sorted[0] - (mids[0] - x_sorted[0])], mids))
        right_edges = np.concatenate((mids, [x_sorted[-1] + (x_sorted[-1] - mids[-1])]))
    else:
        left_edges = np.array([x_sorted[0] - 0.5])
        right_edges = np.array([x_sorted[0] + 0.5])

    spans = []
    i = 0
    n = len(mask)
    while i < n:
        if mask[i]:
            j = i
            while j + 1 < n and mask[j + 1]:
                j += 1
            spans.append((left_edges[i], right_edges[j]))
            i = j + 1
        else:
            i += 1
    return spans


def pl_save_plot(_):
    fig = pl_last_figure.get("fig")
    if fig is None:
        pl_status.value = '<div class="pl-status">No plot is currently available to save.</div>'
        return
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = SAVE_DIR / f"{PROJECT_NAME}_LoadingsBands_{timestamp}.png"
    try:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        pl_last_figure["path"] = str(out_path)
        pl_status.value = f'<div class="pl-status">Plot saved to <b>{out_path}</b></div>'
    except Exception as exc:
        pl_status.value = f'<div class="pl-status">Failed to save plot: {exc}</div>'


def pl_update(_=None):
    pl_plot_output.clear_output(wait=True)
    pl_last_figure["fig"] = None
    pl_last_figure["path"] = None

    folder_path = pl_source_dropdown.value
    if folder_path is None:
        pl_status.value = '<div class="pl-status">No source folder selected.</div>'
        return

    folder_name = Path(folder_path).name
    meta_df = _pl_folder_metadata(folder_path)
    if meta_df.empty:
        pl_status.value = f'<div class="pl-status">No parseable spectra found in {folder_name}.</div>'
        return

    catalog = _pl_filter(meta_df, materials=pl_material.value, regions=pl_region.value,
                         objects=pl_object.value, filename=pl_filename.value,
                         gt_classes=pl_gt_class.value)
    if catalog.empty:
        pl_status.value = '<div class="pl-status">No spectra match the selected filters.</div>'
        return

    _ensure_palette(catalog["gt_class"].unique())

    try:
        # Fit PCA loadings on the full folder (region-filtered), independent of
        # the material/object/GT filter used for the spectra lines.
        wn_ref, loadings, evr, max_pcs = _pl_fit_loadings(folder_path, pl_region.value, pl_n_pcs.value)

        selected_pcs = sorted(set(int(v) for v in pl_pc_select.value if int(v) <= max_pcs))
        if not selected_pcs:
            selected_pcs = [1]
        dropped = sorted(set(int(v) for v in pl_pc_select.value) - set(selected_pcs))
        threshold_frac = pl_threshold.value

        with pl_plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(_FW, _FW * 0.55), constrained_layout=True)

            # --- Threshold bands, one colour per PC, behind the spectra ---
            band_handles = []
            for pc_num in selected_pcs:
                pc_idx = pc_num - 1
                color = _PL_PC_CMAP(pc_idx % 10)
                spans = _pl_threshold_spans(wn_ref, loadings[pc_idx], threshold_frac)
                for x0, x1 in spans:
                    ax.axvspan(x0, x1, color=color, alpha=0.28, zorder=1, linewidth=0)
                band_handles.append(
                    Patch(facecolor=color, alpha=0.28,
                          label=f"PC{pc_num} ≥ {threshold_frac:.0%} max ({evr[pc_idx]*100:.1f}% var)")
                )

            # --- De-emphasised spectra (thin lines) on top of the bands ---
            warnings = []
            _stacked = pl_layout.value == "stacked"
            if pl_mode.value == "single":
                cmap = plt.get_cmap("tab20", max(len(catalog), 1))
                _prepared = []
                for idx, (_, row) in enumerate(catalog.iterrows()):
                    try:
                        df = _pl_read_spectrum(row["path"])
                    except Exception as exc:
                        warnings.append(f'{Path(row["path"]).name} failed ({exc})')
                        continue
                    wn = df["wavenumber"].to_numpy(dtype=float)
                    y = df["signal"].to_numpy(dtype=float)
                    color = _pl_color(row.get("gt_class", "unknown"), idx, cmap)
                    label = f'{row.get("gt_class", "unknown")} | {row["filename"]}'
                    _prepared.append((wn, y, color, label))
                if _stacked and _prepared:
                    _off = pl_offset.value if pl_offset.value > 0 else _pl_auto_offset([p[1] for p in _prepared])
                    for _i, (wn, y, color, label) in enumerate(_prepared):
                        _base = _i * _off
                        ax.plot(wn, y + _base, color=color, linewidth=0.7, alpha=0.85, zorder=2)
                        _pl_stacked_label(ax, _base, label, color)
                else:
                    for wn, y, color, label in _prepared:
                        ax.plot(wn, y, color=color, linewidth=0.7, alpha=0.85, label=label, zorder=2)
                n_groups = len(catalog)
            else:
                groups = _pl_build_groups(catalog, pl_grouping.value)
                cmap = plt.get_cmap("tab10", max(len(groups), 1))
                _prepared = []
                for idx, (legend_label, group_df) in enumerate(groups.items()):
                    spectra_list = []
                    for _, row in group_df.iterrows():
                        try:
                            spectra_list.append(_pl_read_spectrum(row["path"]))
                        except Exception as exc:
                            warnings.append(f'{Path(row["path"]).name} failed ({exc})')
                    if not spectra_list:
                        continue
                    ref_x, stack = _pl_align(spectra_list)
                    mean_y, std_y = stack.mean(axis=0), stack.std(axis=0, ddof=0)
                    if pl_grouping.value == "all":
                        line_color, fill_color, fill_alpha = "black", "lightgrey", 0.35
                    else:
                        line_color = fill_color = _pl_color(legend_label, idx, cmap)
                        fill_alpha = 0.10
                    _prepared.append((ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label))
                if _stacked and _prepared:
                    _off = pl_offset.value if pl_offset.value > 0 else _pl_auto_offset([p[1] for p in _prepared])
                    for _i, (ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label) in enumerate(_prepared):
                        _base = _i * _off
                        ax.plot(ref_x, mean_y + _base, color=line_color, linewidth=1.0, alpha=0.9, zorder=2)
                        if pl_show_std.value:
                            ax.fill_between(ref_x, mean_y + _base - std_y, mean_y + _base + std_y, color=fill_color, alpha=fill_alpha, zorder=2)
                        _pl_stacked_label(ax, _base, legend_label, line_color)
                else:
                    for ref_x, mean_y, std_y, line_color, fill_color, fill_alpha, legend_label in _prepared:
                        ax.plot(ref_x, mean_y, color=line_color, linewidth=1.0, alpha=0.9, label=legend_label, zorder=2)
                        if pl_show_std.value:
                            ax.fill_between(ref_x, mean_y - std_y, mean_y + std_y, color=fill_color, alpha=fill_alpha, zorder=2)
                n_groups = len(groups)

            ax.set_xlabel("Wavenumber (1/cm)")
            ax.set_ylabel("Reflectance")
            ax.grid(True, alpha=0.25)
            ax.invert_xaxis()
            if _stacked:
                ax.set_yticks([])

            # combined legend, inside the axes: spectra handles + one patch per PC
            h_spec, l_spec = ax.get_legend_handles_labels()
            l_band = [h.get_label() for h in band_handles]
            ax.legend(h_spec + band_handles, l_spec + l_band, loc="lower right", fontsize=_FONT_SIZE - 1)

            pl_last_figure["fig"] = fig
            plt.show()

        pc_label = ", ".join(f"PC{p}" for p in selected_pcs)
        dropped_text = "" if not dropped else f" (PC{', PC'.join(str(d) for d in dropped)} unavailable — only {max_pcs} PCs fit)"
        warning_text = "" if not warnings else " Warnings: " + "; ".join(warnings[:8])
        pl_status.value = (f'<div class="pl-status">Updated with <b>{n_groups}</b> series '
                           f'from {folder_name}, band threshold {threshold_frac:.0%} for {pc_label}'
                           f'{dropped_text}.{warning_text}</div>')
    except Exception as exc:
        pl_last_figure["fig"] = None
        pl_status.value = f'<div class="pl-status">Visualization update failed: {exc}</div>'


def pl_refresh_filters(_=None):
    folder_path = pl_source_dropdown.value
    has_regions = METADATA["has_regions"]
    if folder_path is None:
        for w in (pl_material, pl_region, pl_gt_class, pl_object):
            w.options = []
            w.value = ()
        pl_filename.options = ["All"]
        pl_filename.value = "All"
        return

    meta_df = _pl_folder_metadata(folder_path)
    if meta_df.empty:
        for w in (pl_material, pl_region, pl_gt_class, pl_object):
            w.options = []
            w.value = ()
        pl_filename.options = ["All"]
        pl_filename.value = "All"
        pl_update()
        return

    materials  = sorted(meta_df["material"].dropna().astype(str).unique().tolist())
    gt_classes = sorted(meta_df["gt_class"].dropna().astype(str).unique().tolist())
    objects    = sorted(meta_df["object_number"].dropna().astype(str).unique().tolist())

    pl_material.options = materials
    pl_material.value = tuple(v for v in pl_material.value if v in materials)

    if has_regions:
        regions = sorted(meta_df["region"].dropna().astype(str).unique().tolist())
        pl_region.options = regions
        pl_region.disabled = False
        pl_region.value = tuple(v for v in pl_region.value if v in regions)
    else:
        pl_region.options = []
        pl_region.value = ()
        pl_region.disabled = True

    pl_object.options = objects
    pl_object.value = tuple(v for v in pl_object.value if v in objects)

    pl_gt_class.options = gt_classes
    pl_gt_class.value = tuple(v for v in pl_gt_class.value if v in gt_classes)

    filtered = _pl_filter(meta_df, materials=pl_material.value, regions=pl_region.value,
                          objects=pl_object.value, filename="All", gt_classes=pl_gt_class.value)
    filenames = ["All"] + sorted(filtered["filename"].dropna().astype(str).unique().tolist())
    pl_filename.options = filenames
    if pl_filename.value not in filenames:
        pl_filename.value = "All"

    if has_regions:
        pl_grouping.options = ["all", "by material", "by region", "by object", "by GT class"]
    else:
        pl_grouping.options = ["all", "by material", "by object", "by GT class"]
        if pl_grouping.value == "by region":
            pl_grouping.value = "by material"

    pl_update()


def pl_random_file(_):
    options = [opt for opt in pl_filename.options if opt != "All"]
    if options:
        pl_filename.value = _pl_random.choice(options)


def pl_refresh_folders(_=None):
    pl_source_dropdown.options = _pl_discover_folders()
    pl_refresh_filters()


def pl_sync_pc_options(_=None):
    """Keep the 'Show PC #' options in sync with the 'PCA fit — # PCs' count."""
    n = pl_n_pcs.value
    options = list(range(1, n + 1))
    prev = tuple(v for v in pl_pc_select.value if v in options)
    pl_pc_select.options = options
    pl_pc_select.value = prev if prev else tuple(options[:min(5, n)])


# --- Widgets ---
_pl_folder_options = _pl_discover_folders()
pl_source_dropdown = widgets.Dropdown(
    options=_pl_folder_options,
    value=_pl_folder_options[0][1] if _pl_folder_options else None,
    description="Source folder", style=_PL_STYLE, layout=widgets.Layout(width="500px"))

_pl_sel_layout = widgets.Layout(width="200px", height="120px")
pl_material = widgets.SelectMultiple(options=[], value=(), description="Material", style=_PL_STYLE, layout=_pl_sel_layout)
pl_region   = widgets.SelectMultiple(options=[], value=(), description="Region", style=_PL_STYLE, layout=_pl_sel_layout,
                                     disabled=not METADATA["has_regions"])
pl_gt_class = widgets.SelectMultiple(options=[], value=(), description="GT class", style=_PL_STYLE, layout=_pl_sel_layout)
pl_object   = widgets.SelectMultiple(options=[], value=(), description="Object number", style=_PL_STYLE, layout=_pl_sel_layout)
pl_filename = widgets.Dropdown(options=["All"], value="All", description="Specific filename", style=_PL_STYLE, layout=widgets.Layout(width="340px"))
pl_mode = widgets.ToggleButtons(options=[("Single View", "single"), ("Aggregated Group View", "aggregated")], value="single", description="Mode", style=_PL_STYLE)

_pl_grouping_opts = ["all", "by material", "by region", "by object", "by GT class"] if METADATA["has_regions"] else ["all", "by material", "by object", "by GT class"]
pl_grouping = widgets.Dropdown(options=_pl_grouping_opts, value="by material", description="Group by", style=_PL_STYLE, layout=widgets.Layout(width="220px"))
pl_layout = widgets.ToggleButtons(options=[("Overlaid", "overlaid"), ("Stacked", "stacked")],
                                  value="overlaid", description="Layout", style=_PL_STYLE)
pl_offset = widgets.FloatText(value=0.0, step=0.5, description="Stack offset (0=auto)",
                              style=_PL_STYLE, layout=widgets.Layout(width="240px"))
pl_show_std = widgets.Checkbox(value=True, description="Show std band")

pl_n_pcs = widgets.BoundedIntText(value=5, min=1, max=20, description="PCA fit — # PCs:",
                                  style=_PL_STYLE, layout=widgets.Layout(width="220px"))
pl_pc_select = widgets.SelectMultiple(options=list(range(1, 6)), value=(1, 2, 3, 4, 5),
                                      description="Show PC(s):", style=_PL_STYLE,
                                      layout=widgets.Layout(width="160px", height="120px"))
pl_threshold = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                   description="Threshold (% of max |loading|):",
                                   style=_PL_STYLE, layout=widgets.Layout(width="360px"),
                                   continuous_update=False, readout_format=".0%")

pl_random_button  = widgets.Button(description="Random", icon="random", layout=widgets.Layout(width="150px", height="34px"))
pl_refresh_button = widgets.Button(description="Refresh folders", icon="refresh", button_style="info", layout=widgets.Layout(width="150px", height="34px"))
pl_save_button    = widgets.Button(description="Save Plot", icon="save", button_style="success", layout=widgets.Layout(width="150px", height="34px"))
pl_status = widgets.HTML('<div class="pl-status">Select a source folder to inspect spectra.</div>')
pl_plot_output = widgets.Output(layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="8px", height="620px"))

pl_source_dropdown.observe(pl_refresh_filters, names="value")
pl_refresh_button.on_click(pl_refresh_folders)
pl_random_button.on_click(pl_random_file)
pl_save_button.on_click(pl_save_plot)
pl_n_pcs.observe(pl_sync_pc_options, names="value")
for w in (pl_material, pl_region, pl_object, pl_gt_class):
    w.observe(pl_refresh_filters, names="value")
for w in (pl_filename, pl_mode, pl_grouping, pl_pc_select, pl_threshold, pl_layout, pl_offset, pl_show_std):
    w.observe(pl_update, names="value")

pl_controls = widgets.VBox([
    widgets.HBox([pl_source_dropdown, pl_refresh_button]),
    widgets.HBox([pl_material, pl_region, pl_gt_class, pl_object]),
    widgets.HBox([pl_filename, pl_random_button]),
    widgets.HBox([pl_mode, pl_grouping]),
    widgets.HBox([pl_layout, pl_offset, pl_show_std]),
    widgets.HBox([pl_n_pcs, pl_pc_select]),
    widgets.HBox([pl_threshold]),
    widgets.HBox([pl_save_button]),
])

pl_panel = widgets.VBox([
    widgets.HTML(
        '<div class="pl-title">PCA Loadings Threshold Bands over Group Spectra</div>'
        '<div class="pl-subtitle">Same filters as Group Spectra View. The PCA is fit on the full '
        'source folder (respecting the Region filter, independent of Material/Object/GT filters) '
        'with "PCA fit — # PCs" components. "Show PC(s)" selects which PCs to overlay; for each, a '
        'vertical band is shaded wherever |loading| is at or above the threshold fraction of that '
        'PC\'s peak magnitude. Bands sit behind the thin spectra lines.</div>'
    ),
    pl_controls, pl_status, pl_plot_output,
], layout=widgets.Layout(width="100%", border="1px solid #d1d5db", padding="12px", margin="8px 0"))

display(pl_panel)
pl_refresh_filters()